In [7]:
import pandas as pd
import openpyxl
import numpy as np
import re
import heapq

# =====================================================
# LOAD INPUT FILE
# =====================================================
df = pd.read_excel("April - test1_or.xlsx")

# =====================================================
# TEXT NORMALIZATION
# =====================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).lower().strip()

def clean_hotel_name(x):
    x = str(x).lower()
    x = x.replace("singapore", "")
    x = x.replace(",", "")
    x = " ".join(x.split())
    return x.strip()

# =====================================================
# AFTERNOON CHECK
# =====================================================
def is_afternoon(window):
    if pd.isna(window):
        return False
    start = str(window).split("-")[0].strip()
    return ("PM" in start) and (not start.startswith("12"))

df["is_afternoon"] = df["pickup_window"].apply(is_afternoon)

# =====================================================
# ZONE CLASSIFICATION
# =====================================================
def classify_zone(place):
    p = clean_text(place)
    if "sentosa" in p:
        return "SENTOSA"
    if "garden" in p or "marina bay sands" in p:
        return "GARDENS_BY_THE_BAY"
    return "OTHER"

df["zone"] = df["To"].apply(classify_zone)

# =====================================================
# GROUP KEY LOGIC
# =====================================================
def build_group_key(row):
    trip = clean_text(row["trip_type"])

    if "pickup" not in trip:
        return row["pickup_window"]

    if not row["is_afternoon"]:
        return row["pickup_window"]

    if row["zone"] in ["SENTOSA", "GARDENS_BY_THE_BAY"]:
        return f"{row['pickup_window']} | {row['zone']}"

    return row["pickup_window"]

df["group_key"] = df.apply(build_group_key, axis=1)

# =====================================================
# BATCH ID ASSIGNMENT
# =====================================================
def assign_batch_id(window_group):
    window_group = window_group.copy()
    trip = clean_text(window_group["trip_type"].iloc[0])
    prefix = "p" if "pickup" in trip else "r"
    codes = pd.factorize(window_group["group_key"])[0] + 1
    window_group["batch_id"] = [f"{prefix}{i}" for i in codes]
    return window_group

df = (
    df.groupby("pickup_window", group_keys=False, sort=True)
      .apply(assign_batch_id)
)

# =====================================================
# LOAD PARKING CONSTRAINT FILE
# =====================================================
parking_df = pd.read_excel("parking_constraint_lat_split_fin1.xlsx")
parking_df["Hotel_names_clean"] = parking_df["Hotel_names"].apply(clean_hotel_name)

# pickup hotel key
df["From_clean"] = df["From"].apply(clean_hotel_name)

# return hotel key (return-trip clustering uses hotel side from To)
df["Return_hotel_clean"] = df["To"].apply(clean_hotel_name)

# destination clean (used later + helper grouping)
df["To_clean"] = df["To"].apply(clean_hotel_name)

# =====================================================
# MERGE PARKING STATUS (pickup side)
# =====================================================
df = df.merge(
    parking_df[["Hotel_names_clean", "Parking_status"]],
    left_on="From_clean",
    right_on="Hotel_names_clean",
    how="left"
)

# =====================================================
# INITIAL VEHICLE ALLOCATION
# =====================================================
def get_vehicle_allocation(row):
    trip = str(row["trip_type"]).lower().strip()

    if "return" in trip:
        return "default"

    if "pickup" in trip:
        status = str(row["Parking_status"]).lower().strip()
        if status == "coach_allowed":
            return "coach"
        else:
            return "comfi"

    return "default"

df["initial_vehicle_allocation"] = df.apply(get_vehicle_allocation, axis=1)

# =====================================================
# DISTANCE/TIME TABLE
# =====================================================
dist_df = pd.read_excel("hotel_distance_time_with_ids_fin1.xlsx")
dist_df["From_clean"] = dist_df["From"].apply(clean_hotel_name)
dist_df["To_clean"]   = dist_df["To"].apply(clean_hotel_name)

def extract_km(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower().replace("km", "").strip()
    try:
        return float(x)
    except:
        return np.nan

dist_df["distance_km"] = dist_df["distance_text"].apply(extract_km)

def duration_to_min_already_minutes(x):
    """
    duration_text is already in minutes like: "11 mins", "1 min"
    Extract numeric part.
    """
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    m = re.search(r"(\d+)", s)
    if not m:
        return np.nan
    return float(m.group(1))

dist_df["duration_min"] = dist_df["duration_text"].apply(duration_to_min_already_minutes)

df["total_pack"] = df["adult"].fillna(0) + df["child"].fillna(0)

# ✅ pickup-only: if batch pax <= 13 => comfi_dest
batch_pax = (
    df.groupby(["pickup_window", "batch_id"])["total_pack"]
      .transform("sum")
      .fillna(0)
      .astype(int)
)
mask_small_batch = (
    df["trip_type"].str.lower().str.contains("pickup", na=False)
    & (batch_pax <= 13)
)
df.loc[mask_small_batch, "initial_vehicle_allocation"] = "comfi_dest"

# ✅ comfi_dest flag (must exist early)
df["is_comfi_dest"] = df["initial_vehicle_allocation"].astype(str).str.lower().str.strip().eq("comfi_dest")

# =====================================================
# HELPER: DESTINATION KEY FOR COMFI_DEST ONLY (DO NOT RENAME ACTUAL DEST)
# =====================================================
df["_dest_key"] = df["To_clean"].astype(str)

mask_cd_for_key = (
    df["trip_type"].astype(str).str.lower().str.contains("pickup", na=False)
    & df["is_comfi_dest"]
    & df["Date"].notna()
    & df["pickup_window"].notna()
    & df["batch_id"].notna()
)

if mask_cd_for_key.any():
    tmp = df.loc[mask_cd_for_key, ["Date", "pickup_window", "batch_id", "To_clean"]].copy()
    tmp["To_clean"] = tmp["To_clean"].astype(str)

    canon = (
        tmp.groupby(["Date", "pickup_window", "batch_id"])["To_clean"]
           .agg(lambda s: (s.mode().iloc[0] if not s.mode().empty else s.iloc[0]))
           .reset_index()
           .rename(columns={"To_clean": "_dest_key"})
    )

    df = df.merge(canon, on=["Date", "pickup_window", "batch_id"], how="left", suffixes=("", "_canon"))
    df["_dest_key"] = np.where(
        mask_cd_for_key,
        df["_dest_key_canon"].fillna(df["To_clean"].astype(str)),
        df["To_clean"].astype(str)
    )
    df.drop(columns=["_dest_key_canon"], inplace=True, errors="ignore")

# =====================================================
# USER INPUT VEHICLES (COACH + COMFI)
# =====================================================
def parse_vehicle_pool(user_input: str, prefix: str):
    """
    Parse input like:
      2x23, 1x49, 1x55
      6x13, 2x6, 3x4
    Accepts also: 2*23, 2-23, etc
    Returns list of dicts: [{"id": "...", "cap": int}, ...]
    Sorted ascending by cap (best-fit).
    """
    pool = []
    counter = 1

    for part in str(user_input).split(","):
        part = part.strip().lower()
        if not part:
            continue
        part = part.replace("*", "x").replace("-", "x").replace(" ", "")
        m = re.match(r"(\d+)x(\d+)", part)
        if not m:
            print(f"Skipping invalid entry: {part}")
            continue

        count = int(m.group(1))
        cap = int(m.group(2))
        for _ in range(count):
            pool.append({"id": f"{prefix}{counter:03d}", "cap": cap})
            counter += 1

    pool = sorted(pool, key=lambda x: x["cap"])
    return pool

print("\nEnter available coaches like:")
print("Example => 2x23, 1x45, 1x49, 1x55")
coach_input = input("\nEnter coach availability: ").strip()
master_coach_pool = parse_vehicle_pool(coach_input, prefix="coach")
if not master_coach_pool:
    raise ValueError("No valid coach capacity entered. Please rerun.")
print("\nCoach pool template (per pickup window):",
      [f"{c['id']}({c['cap']})" for c in master_coach_pool])

print("\nEnter available COMFI like:")
print("Example => 6x13, 2x6, 3x4")
comfi_input = input("\nEnter COMFI availability: ").strip()
master_comfi_pool = parse_vehicle_pool(comfi_input, prefix="comfi")
if not master_comfi_pool:
    raise ValueError("No valid COMFI capacity entered. Please rerun.")
print("\nCOMFI pool template (per pickup window):",
      [f"{c['id']}({c['cap']})" for c in master_comfi_pool])

# =====================================================
# VEHICLE CAPACITY LOOKUP (optimized_vehicle_id -> cap)
# =====================================================
vehicle_cap_lookup = {}
for v in master_coach_pool:
    vehicle_cap_lookup[str(v["id"])] = int(v["cap"])
for v in master_comfi_pool:
    vehicle_cap_lookup[str(v["id"])] = int(v["cap"])

COACH_DEFAULT_SIZES = [23, 45, 49, 55]
COMFI_DEFAULT_SIZES = [13, 6, 4]

def _best_combo(req, sizes, max_vehicles=6):
    req = int(req)
    sizes = sorted([int(s) for s in sizes])
    max_size = max(sizes)
    max_sum = req + max_size * (max_vehicles - 1)

    dp = {0: (0, {})}
    for _ in range(max_vehicles):
        new_dp = dict(dp)
        for ssum, (nv, combo) in dp.items():
            for sz in sizes:
                nsum = ssum + sz
                if nsum > max_sum:
                    continue
                ncombo = dict(combo)
                ncombo[sz] = ncombo.get(sz, 0) + 1
                cand = (nv + 1, ncombo)
                if nsum not in new_dp:
                    new_dp[nsum] = cand
                else:
                    cur_nv, _cur_combo = new_dp[nsum]
                    if cand[0] < cur_nv:
                        new_dp[nsum] = cand
        dp = new_dp

    best = None
    for ssum, (nv, combo) in dp.items():
        if ssum < req:
            continue
        waste = ssum - req
        big_pref = tuple([-combo.get(sz, 0) for sz in sorted(sizes, reverse=True)])
        cand_key = (waste, nv, big_pref)
        if best is None or cand_key < best[0]:
            best = (cand_key, ssum, nv, combo)

    if best is None:
        mx = max_size
        n = (req + mx - 1) // mx
        return {mx: n}

    return best[3]

def _combo_to_text(combo_dict):
    items = sorted(combo_dict.items(), key=lambda x: (-x[0], x[1]))
    parts = []
    total = 0
    for sz, cnt in items:
        total += int(sz) * int(cnt)
        parts.append(f"{cnt}x{sz}")
    return " + ".join(parts), total

def suggest_vehicle_for_group(req, kind):
    req = int(req)
    sizes = COACH_DEFAULT_SIZES if kind == "coach" else COMFI_DEFAULT_SIZES
    combo = _best_combo(req, sizes, max_vehicles=6)
    combo_txt, total_cap = _combo_to_text(combo)
    cap_string = combo_txt.replace(" + ", "+")
    return f"{combo_txt} ({total_cap} cap)", cap_string

def format_required_label(req, kind):
    sug_txt, _cap_str = suggest_vehicle_for_group(req, kind)
    tag = "COACH" if kind == "coach" else "COMFI"
    return f"{tag} REQUIRED ({int(req)}) [suggest: {sug_txt}]"

def format_required_cap(req, kind):
    _sug_txt, cap_str = suggest_vehicle_for_group(req, kind)
    return cap_str

# =====================================================
# HELPERS (shared)
# =====================================================
def assert_window_capacity_feasible(window_group, coach_pool):
    trip = clean_text(window_group["trip_type"].iloc[0])
    if "pickup" not in trip:
        return

    alloc = window_group["initial_vehicle_allocation"].astype(str).str.lower().str.strip()
    coach_rows = window_group[alloc.eq("coach")]

    total_pax = int(coach_rows["total_pack"].fillna(0).sum())
    total_cap = int(sum(int(c["cap"]) for c in coach_pool))
    pw = str(window_group["pickup_window"].iloc[0])

    print(f"\n[CAPACITY CHECK] pickup_window='{pw}' coach_pax={total_pax} vs total_coach_capacity={total_cap}")
    if total_pax > total_cap:
        raise ValueError(
            f"INSUFFICIENT COACH CAPACITY for pickup_window='{pw}': "
            f"coach_pax={total_pax} > total_coach_capacity={total_cap}."
        )

def pick_best_fit_vehicle(pool, pax_required):
    pax_required = int(pax_required)
    for i, v in enumerate(pool):
        if int(v["cap"]) >= pax_required:
            return v, pool[:i] + pool[i+1:]
    return None, pool

def group_pax(gset, packs):
    return int(sum(packs.get(h, 0) for h in gset))

def build_weighted_graph(hotels):
    hotels = set(hotels)
    adj_w = {h: {} for h in hotels}

    sub = dist_df[
        (dist_df["From_clean"].isin(hotels)) &
        (dist_df["To_clean"].isin(hotels)) &
        (dist_df["distance_km"].notna())
    ][["From_clean", "To_clean", "distance_km"]]

    for a, b, km in sub.itertuples(index=False, name=None):
        if b not in adj_w[a] or km < adj_w[a][b]:
            adj_w[a][b] = km
        if a not in adj_w[b] or km < adj_w[b][a]:
            adj_w[b][a] = km

    return adj_w

def build_adj_within_radius(adj_w, radius_km):
    adj = {h: set() for h in adj_w}
    for a in adj_w:
        for b, km in adj_w[a].items():
            if km <= radius_km:
                adj[a].add(b)
    return adj

def dijkstra(adj_w, src):
    dist = {src: 0.0}
    prev = {}
    pq = [(0.0, src)]
    seen = set()

    while pq:
        d, u = heapq.heappop(pq)
        if u in seen:
            continue
        seen.add(u)

        for v, w in adj_w.get(u, {}).items():
            nd = d + float(w)
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))

    return dist, prev

def min_edge_to_group(candidate, group_nodes, adj_w):
    best = np.inf
    best_to = None
    for g in group_nodes:
        d = adj_w.get(candidate, {}).get(g, np.inf)
        if d < best:
            best = d
            best_to = g
    return best, best_to

def choose_next_by_distance_that_keeps_feasible(current_group, frontier, current_pax, packs, coach_pool, adj_w, print_distances):
    scored = []
    for cand in frontier:
        d, to = min_edge_to_group(cand, current_group, adj_w)
        scored.append((d, cand, to))
    scored.sort(key=lambda x: (x[0], str(x[1])))

    for d, cand, to in scored:
        cand_pax = int(packs.get(cand, 0))
        new_pax = current_pax + cand_pax
        feasible = any(int(c["cap"]) >= new_pax for c in coach_pool)

        if print_distances:
            print(f"  Consider add '{cand}' pax={cand_pax} closest_to='{to}' dist={d:.3f} km "
                  f"| current_pax={current_pax} -> would_be={new_pax} | feasible_with_remaining_pool={feasible}")

        if feasible:
            return cand, d, to, new_pax

    return None, np.inf, None, current_pax

def to_excel_letters(n: int) -> str:
    s = ""
    while n > 0:
        n, r = divmod(n - 1, 26)
        s = chr(97 + r) + s
    return s

# =====================================================
# STEP 2 (PICKUP) — COMFI_DEST
# =====================================================
def build_step2_batches_for_one_batch_COMFI_DEST(
    group,
    comfi_pool,
    pickup_window_val,
    radius_km=1.5,
    print_distances=True
):
    group = group.copy()
    batch_prefix = str(group["batch_id"].iloc[0])

    hotel_summary = (
        group.groupby("From_clean", dropna=False)
             .agg(total_pack=("total_pack", "sum"))
             .reset_index()
    )
    packs = dict(zip(hotel_summary["From_clean"], hotel_summary["total_pack"]))
    hotels = hotel_summary["From_clean"].tolist()
    total_batch_pax = int(sum(int(packs.get(h, 0)) for h in hotels))

    adj_w = build_weighted_graph(hotels)
    adj_r = build_adj_within_radius(adj_w, radius_km)

    if print_distances:
        print(
            f"\n[STEP2 COMFI_DEST ONLY] pickup_window='{pickup_window_val}' "
            f"batch_id='{batch_prefix}' total_pax={total_batch_pax}"
        )

    def _alloc(pax_needed):
        nonlocal comfi_pool
        veh, comfi_pool_new = pick_best_fit_vehicle(comfi_pool, int(pax_needed))
        if veh is None:
            return None
        comfi_pool = comfi_pool_new
        return veh

    formed_groups = []
    group_vehicle = []
    group_seed = []

    veh_all = _alloc(total_batch_pax)
    if veh_all is not None:
        seed = max(hotels, key=lambda h: (int(packs.get(h, 0)), str(h)))
        formed_groups = [set(hotels)]
        group_vehicle = [veh_all]
        group_seed = [seed]
    else:
        if print_distances:
            print(f"  No single COMFI fits total_pax={total_batch_pax}. Splitting...")

        unassigned = set(hotels)

        while unassigned:
            seed = max(unassigned, key=lambda h: (int(packs.get(h, 0)), str(h)))
            current = {seed}
            current_pax = int(packs.get(seed, 0))
            unassigned.remove(seed)

            if print_distances:
                print(f"  Start split seed='{seed}' pax={current_pax}")

            frontier = (set(adj_r.get(seed, set())) & unassigned)

            while True:
                nxt, d, to, new_pax = choose_next_by_distance_that_keeps_feasible(
                    current_group=current,
                    frontier=frontier,
                    current_pax=current_pax,
                    packs=packs,
                    coach_pool=comfi_pool,
                    adj_w=adj_w,
                    print_distances=print_distances
                )
                if nxt is None:
                    break
                current.add(nxt)
                current_pax = new_pax
                unassigned.remove(nxt)
                frontier.discard(nxt)
                frontier |= (set(adj_r.get(nxt, set())) & unassigned)

            veh = _alloc(current_pax)
            if veh is None:
                ordered = sorted(list(current), key=lambda h: (-int(packs.get(h, 0)), str(h)))
                if print_distances:
                    print(f"    Still no COMFI fits cluster_pax={current_pax}. Trying per-hotel...")

                for h in ordered:
                    hp = int(packs.get(h, 0))
                    vehh = _alloc(hp)
                    formed_groups.append({h})
                    group_vehicle.append(vehh if vehh is not None else {"id": None, "cap": hp})
                    group_seed.append(h)
            else:
                formed_groups.append(set(current))
                group_vehicle.append(veh)
                group_seed.append(seed)

    hotel_to_sid = {}
    sid_to_vehicle = {}
    sid_to_seed = {}

    step_counter = 1
    for gi, gset in enumerate(formed_groups):
        sid = f"{batch_prefix}-{step_counter}"
        step_counter += 1
        for h in gset:
            hotel_to_sid[h] = sid
        sid_to_vehicle[sid] = group_vehicle[gi]
        sid_to_seed[sid] = group_seed[gi]

    group["step2_batch_id"] = group["From_clean"].map(hotel_to_sid)

    group["vehicle_id"] = group["step2_batch_id"].map(
        lambda sid: sid_to_vehicle[sid]["id"] if sid_to_vehicle.get(sid) and sid_to_vehicle[sid].get("id") else None
    )
    group["vehicle_capacity"] = group["step2_batch_id"].map(
        lambda sid: sid_to_vehicle[sid]["cap"] if sid_to_vehicle.get(sid) and sid_to_vehicle[sid].get("cap") else None
    )
    group["vehicle_label"] = group.apply(
        lambda r: (
            f"{r['vehicle_id']} ({int(r['vehicle_capacity'])} seater)"
            if pd.notna(r["vehicle_id"]) and pd.notna(r["vehicle_capacity"])
            else None
        ),
        axis=1
    )
    group["assigned_vehicle_capacity"] = group["vehicle_capacity"]

    hotel_rank_map = {}
    hotel_path_map = {}

    for sid, seed in sid_to_seed.items():
        sid_hotels = [h for h, s in hotel_to_sid.items() if s == sid]
        sid_set = set(sid_hotels)
        if not sid_hotels:
            continue
        if seed not in sid_set:
            seed = sorted(sid_hotels)[0]

        chain = [seed]
        remaining = sid_set - {seed}
        current = seed

        while remaining:
            dist_map, _prev = dijkstra(adj_w, current)
            best_next = None
            best_dist = np.inf
            for cand in remaining:
                d = dist_map.get(cand, np.inf)
                if (d < best_dist) or (d == best_dist and str(cand) < str(best_next)):
                    best_dist = d
                    best_next = cand
            if best_next is None or np.isinf(best_dist):
                best_next = sorted(list(remaining))[0]
            chain.append(best_next)
            remaining.remove(best_next)
            current = best_next

        for i, h in enumerate(chain, start=1):
            hotel_rank_map[h] = to_excel_letters(i)
            hotel_path_map[h] = " -> ".join(chain[:i])

    group["rank"] = group["From_clean"].map(hotel_rank_map)
    group["path_from_seed"] = group["From_clean"].map(hotel_path_map)

    return group, comfi_pool

# =====================================================
# STEP 2 (PICKUP) — COACH
# =====================================================
def build_step2_batches_for_one_batch_COACH(group, coach_pool, pickup_window_val, radius_km=1.5, print_distances=True, small_batch_cap=13):
    group = group.copy()
    batch_prefix = group["batch_id"].iloc[0]

    total_batch_pax = int(group["total_pack"].fillna(0).sum())
    if total_batch_pax <= int(small_batch_cap):
        sid = f"{batch_prefix}-1"
        group["step2_batch_id"] = sid

        group["vehicle_id"] = None
        group["vehicle_capacity"] = None
        group["vehicle_label"] = None
        group["assigned_vehicle_capacity"] = None

        hotels = sorted(group["From_clean"].dropna().astype(str).unique().tolist())
        hotel_rank_map = {}
        hotel_path_map = {}
        for i, h in enumerate(hotels, start=1):
            hotel_rank_map[h] = to_excel_letters(i)
            hotel_path_map[h] = " -> ".join(hotels[:i])

        group["rank"] = group["From_clean"].astype(str).map(hotel_rank_map)
        group["path_from_seed"] = group["From_clean"].astype(str).map(hotel_path_map)

        if print_distances:
            print(
                f"\n[STEP2 SMALL-BATCH] pickup_window='{pickup_window_val}' "
                f"batch_id='{batch_prefix}' total_pax={total_batch_pax} "
                f"<= {small_batch_cap} => single step2_batch_id='{sid}'"
            )

        return group, coach_pool

    hotel_summary = (
        group.groupby("From_clean", dropna=False)
             .agg(total_pack=("total_pack", "sum"))
             .reset_index()
    )
    packs = dict(zip(hotel_summary["From_clean"], hotel_summary["total_pack"]))
    hotels = hotel_summary["From_clean"].tolist()

    adj_w = build_weighted_graph(hotels)
    adj_r = build_adj_within_radius(adj_w, radius_km)

    unassigned = set(hotels)

    formed_groups, group_coach, group_caps, group_seed = [], [], [], []
    singletons = []

    while unassigned:
        seed = max(unassigned, key=lambda h: (packs.get(h, 0), str(h)))
        current = {seed}
        current_pax = int(packs.get(seed, 0))
        unassigned.remove(seed)

        if print_distances:
            print(f"\n[BATCH {batch_prefix} | {pickup_window_val}] Start cluster seed='{seed}' seed_pax={current_pax}")

        frontier = (set(adj_r.get(seed, set())) & unassigned)

        while True:
            nxt, d, to, new_pax = choose_next_by_distance_that_keeps_feasible(
                current_group=current,
                frontier=frontier,
                current_pax=current_pax,
                packs=packs,
                coach_pool=coach_pool,
                adj_w=adj_w,
                print_distances=print_distances
            )
            if nxt is None:
                break
            current.add(nxt)
            current_pax = new_pax
            unassigned.remove(nxt)
            frontier.discard(nxt)
            frontier |= (set(adj_r.get(nxt, set())) & unassigned)

        if len(current) == 1:
            singletons.append(seed)
            continue

        coach, coach_pool_new = pick_best_fit_vehicle(coach_pool, current_pax)
        if coach is None:
            raise ValueError(
                f"No remaining coach can fit group pax={current_pax} in batch_id='{batch_prefix}' "
                f"pickup_window='{pickup_window_val}'. Remaining caps={[c['cap'] for c in coach_pool]}"
            )

        coach_pool = coach_pool_new
        formed_groups.append(current)
        group_coach.append(coach)
        group_caps.append(int(coach["cap"]))
        group_seed.append(seed)

    if singletons and formed_groups:
        singletons = sorted(singletons, key=lambda h: (-packs.get(h, 0), str(h)))
        while singletons:
            h = singletons[0]
            h_pax = int(packs.get(h, 0))
            dist_map, _prev = dijkstra(adj_w, h)

            candidates = []
            for gi, gset in enumerate(formed_groups):
                best_dist = np.inf
                for node in gset:
                    best_dist = min(best_dist, dist_map.get(node, np.inf))
                candidates.append((best_dist, gi))
            candidates.sort()

            attached = False
            for _d, gi in candidates:
                cap = group_caps[gi]
                new_pax = group_pax(formed_groups[gi], packs) + h_pax
                if new_pax <= cap:
                    formed_groups[gi].add(h)
                    singletons.pop(0)
                    attached = True
                    break

            if attached:
                continue

            others = singletons[1:]
            if not others:
                break

            best_partner = None
            best_partner_dist = np.inf
            for x in others:
                d = dist_map.get(x, np.inf)
                if d < best_partner_dist:
                    best_partner_dist = d
                    best_partner = x

            if best_partner is None or np.isinf(best_partner_dist):
                singletons.pop(0)
                singletons.append(h)
                if len(singletons) == 1:
                    break
                continue

            pair_pax = h_pax + int(packs.get(best_partner, 0))
            coach, coach_pool_new = pick_best_fit_vehicle(coach_pool, pair_pax)
            if coach is None:
                singletons.pop(0)
                singletons.append(h)
                if len(singletons) == 1:
                    break
                continue

            coach_pool = coach_pool_new
            formed_groups.append({h, best_partner})
            group_coach.append(coach)
            group_caps.append(int(coach["cap"]))
            group_seed.append(h)

            singletons.pop(0)
            singletons.remove(best_partner)

    hotel_to_sid, sid_to_coach, sid_to_seed = {}, {}, {}
    step_counter = 1
    for gi, gset in enumerate(formed_groups):
        sid = f"{batch_prefix}-{step_counter}"
        step_counter += 1
        for h in gset:
            hotel_to_sid[h] = sid
        sid_to_coach[sid] = group_coach[gi]
        sid_to_seed[sid] = group_seed[gi]

    for h in singletons:
        sid = f"{batch_prefix}-{step_counter}"
        step_counter += 1
        hotel_to_sid[h] = sid
        sid_to_coach[sid] = None
        sid_to_seed[sid] = h

    group["step2_batch_id"] = group["From_clean"].map(hotel_to_sid)

    group["vehicle_id"] = group["step2_batch_id"].map(
        lambda sid: sid_to_coach[sid]["id"] if sid_to_coach.get(sid) else None
    )
    group["vehicle_capacity"] = group["step2_batch_id"].map(
        lambda sid: sid_to_coach[sid]["cap"] if sid_to_coach.get(sid) else None
    )
    group["vehicle_label"] = group.apply(
        lambda r: f"{r['vehicle_id']} ({int(r['vehicle_capacity'])} seater)"
        if pd.notna(r["vehicle_id"]) and pd.notna(r["vehicle_capacity"]) else None,
        axis=1
    )
    group["assigned_vehicle_capacity"] = group["vehicle_capacity"]

    hotel_rank_map = {}
    hotel_path_map = {}

    for sid, seed in sid_to_seed.items():
        sid_hotels = [h for h, s in hotel_to_sid.items() if s == sid]
        sid_set = set(sid_hotels)
        if not sid_hotels:
            continue
        if seed not in sid_set:
            seed = sorted(sid_hotels)[0]

        chain = [seed]
        remaining = sid_set - {seed}
        current = seed

        while remaining:
            dist_map, _prev = dijkstra(adj_w, current)
            best_next = None
            best_dist = np.inf
            for cand in remaining:
                d = dist_map.get(cand, np.inf)
                if (d < best_dist) or (d == best_dist and str(cand) < str(best_next)):
                    best_dist = d
                    best_next = cand
            if best_next is None or np.isinf(best_dist):
                best_next = sorted(list(remaining))[0]
            chain.append(best_next)
            remaining.remove(best_next)
            current = best_next

        for i, h in enumerate(chain, start=1):
            hotel_rank_map[h] = to_excel_letters(i)
            hotel_path_map[h] = " -> ".join(chain[:i])

    group["rank"] = group["From_clean"].map(hotel_rank_map)
    group["path_from_seed"] = group["From_clean"].map(hotel_path_map)

    return group, coach_pool

# =====================================================
# STEP 2 (RETURN) — COACH ONLY split by capacity
# =====================================================
def build_step2_batches_for_one_return_batch_COACH_ONLY(
    group,
    coach_pool,
    pickup_window_val,
    radius_km=1.5,
    print_distances=True
):
    group = group.copy()
    batch_prefix = str(group["batch_id"].iloc[0])

    hotel_summary = (
        group.groupby("Return_hotel_clean", dropna=False)
             .agg(total_pack=("total_pack", "sum"))
             .reset_index()
    )
    packs = dict(zip(hotel_summary["Return_hotel_clean"], hotel_summary["total_pack"]))
    hotels = [
        h for h in hotel_summary["Return_hotel_clean"].tolist()
        if str(h).strip() and str(h).lower() not in ("nan", "none")
    ]

    if not hotels:
        group["step2_batch_id"] = None
        group["vehicle_id"] = None
        group["vehicle_capacity"] = None
        group["vehicle_label"] = None
        group["assigned_vehicle_capacity"] = None
        group["rank"] = None
        group["path_from_seed"] = None
        return group, coach_pool

    total_batch_pax = int(sum(int(packs.get(h, 0)) for h in hotels))

    if print_distances:
        print(
            f"\n[STEP2 RETURN COACH-ONLY] pickup_window='{pickup_window_val}' "
            f"batch_id='{batch_prefix}' total_pax={total_batch_pax} hotels={len(hotels)}"
        )

    coach_all, coach_pool_new = pick_best_fit_vehicle(coach_pool, total_batch_pax)
    if coach_all is not None:
        sid = f"{batch_prefix}-1"
        group["step2_batch_id"] = sid
        group["vehicle_id"] = coach_all["id"]
        group["vehicle_capacity"] = int(coach_all["cap"])
        group["vehicle_label"] = f"{coach_all['id']} ({int(coach_all['cap'])} seater)"
        group["assigned_vehicle_capacity"] = group["vehicle_capacity"]
        group["rank"] = None
        group["path_from_seed"] = None

        if print_distances:
            print(f"  ✅ Single coach fits total_pax={total_batch_pax} -> assign {coach_all['id']} to whole batch")

        return group, coach_pool_new

    items = sorted([(h, int(packs.get(h, 0))) for h in hotels], key=lambda x: (-x[1], str(x[0])))

    formed_groups, group_vehicle, group_req = [], [], []

    coach_pool = sorted(coach_pool, key=lambda x: int(x["cap"]))

    while items:
        if not coach_pool:
            remaining_pax = sum(p for _, p in items)
            formed_groups.append(set(h for h, _ in items))
            group_vehicle.append(None)
            group_req.append(int(remaining_pax))
            items = []
            break

        coach = coach_pool.pop()  # largest
        cap = int(coach["cap"])

        grp_hotels = []
        used = 0
        i = 0
        while i < len(items):
            h, pax = items[i]
            if used + pax <= cap:
                grp_hotels.append(h)
                used += pax
                items.pop(i)
            else:
                i += 1

        if not grp_hotels:
            h, pax = items.pop(0)
            if pax > cap:
                formed_groups.append({h})
                group_vehicle.append(None)
                group_req.append(int(pax))
                continue
            grp_hotels = [h]
            used = pax

        formed_groups.append(set(grp_hotels))
        group_vehicle.append(coach)
        group_req.append(0)

    hotel_to_sid = {}
    sid_to_vehicle = {}
    sid_to_req = {}

    step_counter = 1
    for gi, gset in enumerate(formed_groups):
        sid = f"{batch_prefix}-{step_counter}"
        step_counter += 1
        for h in gset:
            hotel_to_sid[h] = sid
        sid_to_vehicle[sid] = group_vehicle[gi]
        sid_to_req[sid] = group_req[gi]

    group["step2_batch_id"] = group["Return_hotel_clean"].map(hotel_to_sid)

    group["vehicle_id"] = group["step2_batch_id"].map(
        lambda sid: sid_to_vehicle[sid]["id"] if sid_to_vehicle.get(sid) else None
    )
    group["vehicle_capacity"] = group["step2_batch_id"].map(
        lambda sid: sid_to_vehicle[sid]["cap"] if sid_to_vehicle.get(sid) else None
    )

    group["vehicle_label"] = group.apply(
        lambda r: (
            f"{r['vehicle_id']} ({int(r['vehicle_capacity'])} seater)"
            if pd.notna(r["vehicle_id"]) and pd.notna(r["vehicle_capacity"])
            else f"REQUIRED ({int(sid_to_req.get(str(r['step2_batch_id']), 0))})"
                 if pd.notna(r["step2_batch_id"]) and int(sid_to_req.get(str(r["step2_batch_id"]), 0)) > 0
                 else None
        ),
        axis=1
    )

    group["assigned_vehicle_capacity"] = group["vehicle_capacity"]
    group["rank"] = None
    group["path_from_seed"] = None

    return group, coach_pool

# =====================================================
# INIT STEP2 OUTPUT COLS
# =====================================================
for col in ["step2_batch_id", "assigned_vehicle_capacity", "vehicle_id", "vehicle_capacity",
            "vehicle_label", "rank", "path_from_seed"]:
    df[col] = None

# =====================================================
# APPLY STEP2 (PICKUP)
# =====================================================
pickup_df = df[df["trip_type"].str.lower().str.contains("pickup", na=False)].copy()

def process_one_pickup_window(window_group):
    coach_pool = master_coach_pool.copy()
    comfi_pool = master_comfi_pool.copy()

    assert_window_capacity_feasible(window_group, coach_pool)
    pickup_window_val = str(window_group["pickup_window"].iloc[0])

    outputs = []
    for _batch_id, grp in window_group.groupby("batch_id", sort=True):
        allocs = grp["initial_vehicle_allocation"].astype(str).str.lower().str.strip()

        if (allocs == "comfi_dest").any():
            grp2, comfi_pool = build_step2_batches_for_one_batch_COMFI_DEST(
                grp,
                comfi_pool=comfi_pool,
                pickup_window_val=pickup_window_val,
                radius_km=1.5,
                print_distances=True
            )
            outputs.append(grp2)
        else:
            grp2, coach_pool = build_step2_batches_for_one_batch_COACH(
                grp,
                coach_pool=coach_pool,
                pickup_window_val=pickup_window_val,
                radius_km=1.5,
                print_distances=True
            )
            outputs.append(grp2)

    return pd.concat(outputs, ignore_index=False)

pickup_df = (
    pickup_df.groupby("pickup_window", group_keys=False, sort=True)
             .apply(process_one_pickup_window)
)
df.update(pickup_df)

# =====================================================
# ✅ POST STEP-2 PROMOTION: if step2_batch_id pax <= 13 => comfi_dest
#    (this fixes cases like p1-4)
# =====================================================
mask_pick_step2 = (
    df["trip_type"].astype(str).str.lower().str.contains("pickup", na=False)
    & df["step2_batch_id"].notna()
    & df["Date"].notna()
)

# pax at step2_batch_id level (per day + window + step2 id)
step2_pax = (
    df.groupby(["Date", "pickup_window", "step2_batch_id"])["total_pack"]
      .transform("sum")
      .fillna(0)
      .astype(int)
)

mask_promote_cd = mask_pick_step2 & (step2_pax <= 13)

# Promote these Step-2 groups to comfi_dest
df.loc[mask_promote_cd, "initial_vehicle_allocation"] = "comfi_dest"
df.loc[mask_promote_cd, "is_comfi_dest"] = True

# IMPORTANT: remove coach allocation fields (otherwise later logic will think it's a 45-seater comfi_dest)
for col in ["vehicle_id", "vehicle_capacity", "vehicle_label", "assigned_vehicle_capacity"]:
    if col in df.columns:
        df.loc[mask_promote_cd, col] = None


# =====================================================
# APPLY STEP2 (RETURN) — COACH ONLY
# =====================================================
return_df = df[df["trip_type"].str.lower().str.contains("return", na=False)].copy()

def process_one_return_window(window_group):
    coach_pool = master_coach_pool.copy()
    pw = str(window_group["pickup_window"].iloc[0])

    outs = []
    for _bid, grp in window_group.groupby("batch_id", sort=True):
        grp2, coach_pool = build_step2_batches_for_one_return_batch_COACH_ONLY(
            grp,
            coach_pool=coach_pool,
            pickup_window_val=pw,
            radius_km=1.5,
            print_distances=True
        )
        outs.append(grp2)

    return pd.concat(outs, ignore_index=False) if outs else window_group

if not return_df.empty:
    return_df = (
        return_df.groupby("pickup_window", group_keys=False, sort=True)
                 .apply(process_one_return_window)
    )
    df.update(return_df)

# =====================================================
# STEP 2 VEHICLE BATCH (coach vs comfi + meta columns) — PICKUP ONLY
# =====================================================
WAIT_MIN_PER_STOP = 7
COACH_MAX_ROUTE_MIN = 30

dur_direct = {}
sub_time = dist_df[dist_df["duration_min"].notna()][["From_clean", "To_clean", "duration_min"]]
for a, b, t in sub_time.itertuples(index=False, name=None):
    key = (a, b)
    t = float(t)
    if key not in dur_direct or t < dur_direct[key]:
        dur_direct[key] = t

def tmin(a, b):
    return dur_direct.get((a, b), np.inf)

def hotel_to_hotel_min(a, b):
    if a == b:
        return 0.0
    d = tmin(a, b)
    if np.isinf(d):
        d = tmin(b, a)
    return d

def dest_to_hotel_min(dest, hotel):
    return hotel_to_hotel_min(dest, hotel)

def chain_time_ok(total_time):
    return total_time <= COACH_MAX_ROUTE_MIN + 1e-9

def build_chain_from_anchor(anchor, eligible, max_pack_hotel):
    chain = [anchor]
    chain_set = {anchor}
    remaining = set(eligible) - {anchor}

    total_time = WAIT_MIN_PER_STOP
    current = anchor

    while remaining:
        if (max_pack_hotel not in chain_set) and (max_pack_hotel in remaining):
            dmax = hotel_to_hotel_min(current, max_pack_hotel)
            if not np.isinf(dmax):
                trial = total_time + float(dmax) + WAIT_MIN_PER_STOP
                if chain_time_ok(trial):
                    chain.append(max_pack_hotel)
                    chain_set.add(max_pack_hotel)
                    remaining.remove(max_pack_hotel)
                    total_time = trial
                    current = max_pack_hotel
                    continue

        candidates = []
        for cand in remaining:
            d = hotel_to_hotel_min(current, cand)
            candidates.append((d, str(cand), cand))
        candidates.sort()

        if not candidates:
            break

        d, _nm, nxt = candidates[0]
        if np.isinf(d):
            break

        trial = total_time + float(d) + WAIT_MIN_PER_STOP
        if chain_time_ok(trial):
            chain.append(nxt)
            chain_set.add(nxt)
            remaining.remove(nxt)
            total_time = trial
            current = nxt
        else:
            break

    return chain_set, chain, total_time

def build_coach_chain_must_include_maxpack(hotels_all, forced_comfi, dest, packs_map):
    eligible = [h for h in hotels_all if h not in forced_comfi]
    if not eligible:
        return {
            "coach_set": set(),
            "nearest_hotel": None,
            "nearest_min": np.nan,
            "anchor_used": None,
            "anchor_used_min": np.nan,
            "must_include": None,
            "chain_order": "",
            "chain_time": np.nan,
        }

    must_include = max(eligible, key=lambda h: (packs_map.get(h, 0), str(h)))
    anchors = sorted(eligible, key=lambda h: (dest_to_hotel_min(dest, h), str(h)))

    nearest_hotel = None
    nearest_min = np.inf
    for h in anchors:
        t = dest_to_hotel_min(dest, h)
        if np.isinf(t):
            continue
        nearest_hotel = h
        nearest_min = t
        break
    if nearest_hotel is None:
        nearest_min = np.nan

    best = None

    for anchor in anchors:
        a_min = dest_to_hotel_min(dest, anchor)
        if np.isinf(a_min):
            continue

        coach_set, chain_order, chain_time = build_chain_from_anchor(anchor, eligible, must_include)

        if must_include in coach_set:
            return {
                "coach_set": coach_set,
                "nearest_hotel": nearest_hotel,
                "nearest_min": float(nearest_min) if not np.isnan(nearest_min) else np.nan,
                "anchor_used": anchor,
                "anchor_used_min": float(a_min),
                "must_include": must_include,
                "chain_order": " -> ".join(chain_order),
                "chain_time": float(chain_time),
            }

        score = sum(int(packs_map.get(h, 0)) for h in coach_set)
        if best is None:
            best = (score, chain_time, anchor, a_min, coach_set, chain_order)
        else:
            b_score, b_time, *_ = best
            if (score > b_score) or (score == b_score and chain_time < b_time):
                best = (score, chain_time, anchor, a_min, coach_set, chain_order)

    if best is None:
        return {
            "coach_set": set(),
            "nearest_hotel": nearest_hotel,
            "nearest_min": float(nearest_min) if nearest_hotel else np.nan,
            "anchor_used": None,
            "anchor_used_min": np.nan,
            "must_include": must_include,
            "chain_order": "",
            "chain_time": np.nan,
        }

    _score, b_time, b_anchor, b_anchor_min, b_set, b_chain = best
    return {
        "coach_set": b_set,
        "nearest_hotel": nearest_hotel,
        "nearest_min": float(nearest_min) if nearest_hotel else np.nan,
        "anchor_used": b_anchor,
        "anchor_used_min": float(b_anchor_min),
        "must_include": must_include,
        "chain_order": " -> ".join(b_chain),
        "chain_time": float(b_time),
    }

def last_hotel_from_chain(chain_text: str):
    chain = str(chain_text or "").strip()
    if not chain:
        return None
    parts = [p.strip() for p in chain.split("->") if p.strip()]
    return parts[-1] if parts else None

df["step2_vehicle_batch"] = None
df["nearest_to_destination_hotel"] = None
df["nearest_to_destination_min"] = None
df["coach_anchor_used"] = None
df["coach_chain_order"] = None
df["coach_chain_time_min"] = None
df["coach_must_include_hotel"] = None

mask_pick_all_step2 = (
    df["trip_type"].str.lower().str.contains("pickup", na=False)
    & df["step2_batch_id"].notna()
)

mask_pick_comfi_dest = mask_pick_all_step2 & (df["is_comfi_dest"])
mask_pick_non_cd = mask_pick_all_step2 & (~df["is_comfi_dest"])

df.loc[mask_pick_comfi_dest, "step2_vehicle_batch"] = "comfi_dest"

assign_map = {}
meta_map = {}

pickup_rows = df[mask_pick_non_cd].copy()

for (pw, sid, destk), g in pickup_rows.groupby(["pickup_window", "step2_batch_id", "_dest_key"], dropna=False, sort=False):
    group_key = (str(pw), str(sid), str(destk))

    hotel_packs = g.groupby("From_clean", dropna=False)["total_pack"].sum().to_dict()

    hotels_unique = (
        g[["From_clean", "initial_vehicle_allocation"]]
        .drop_duplicates(subset=["From_clean"])
        .copy()
    )
    hotels_list = hotels_unique["From_clean"].dropna().astype(str).tolist()

    forced_comfi = set(
        hotels_unique.loc[
            hotels_unique["initial_vehicle_allocation"].astype(str).str.lower().str.strip() == "comfi",
            "From_clean"
        ].dropna().astype(str).tolist()
    )

    meta = build_coach_chain_must_include_maxpack(
        hotels_all=hotels_list,
        forced_comfi=forced_comfi,
        dest=str(destk),
        packs_map=hotel_packs
    )
    meta_map[group_key] = meta
    coach_set = meta["coach_set"]

    for h in hotels_list:
        if h in forced_comfi:
            assign_map[(group_key, h)] = "comfi"
        elif h in coach_set:
            assign_map[(group_key, h)] = "coach"
        else:
            assign_map[(group_key, h)] = "comfi"

df.loc[mask_pick_non_cd, "step2_vehicle_batch"] = df.loc[mask_pick_non_cd, ["pickup_window", "step2_batch_id", "_dest_key", "From_clean"]].apply(
    lambda r: assign_map.get(((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), str(r["From_clean"])), "comfi"),
    axis=1
)

mask_force = mask_pick_non_cd & (df["initial_vehicle_allocation"].astype(str).str.lower().str.strip() == "comfi")
df.loc[mask_force, "step2_vehicle_batch"] = "comfi"

df.loc[mask_pick_non_cd, "nearest_to_destination_hotel"] = df.loc[mask_pick_non_cd, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("nearest_hotel"),
    axis=1
)
df.loc[mask_pick_non_cd, "nearest_to_destination_min"] = df.loc[mask_pick_non_cd, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("nearest_min"),
    axis=1
)
df.loc[mask_pick_non_cd, "coach_anchor_used"] = df.loc[mask_pick_non_cd, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("anchor_used"),
    axis=1
)
df.loc[mask_pick_non_cd, "coach_chain_order"] = df.loc[mask_pick_non_cd, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("chain_order"),
    axis=1
)
df.loc[mask_pick_non_cd, "coach_chain_time_min"] = df.loc[mask_pick_non_cd, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("chain_time"),
    axis=1
)
df.loc[mask_pick_non_cd, "coach_must_include_hotel"] = df.loc[mask_pick_non_cd, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("must_include"),
    axis=1
)

cd_meta_map = {}
pickup_cd = df[mask_pick_comfi_dest].copy()

for (pw, sid, destk), g in pickup_cd.groupby(["pickup_window", "step2_batch_id", "_dest_key"], dropna=False, sort=False):
    key = (str(pw), str(sid), str(destk))
    hotel_packs = g.groupby("From_clean", dropna=False)["total_pack"].sum().to_dict()
    hotels_list = (
        g[["From_clean"]].drop_duplicates(subset=["From_clean"])["From_clean"]
        .dropna().astype(str).tolist()
    )
    meta = build_coach_chain_must_include_maxpack(
        hotels_all=hotels_list,
        forced_comfi=set(),
        dest=str(destk),
        packs_map=hotel_packs
    )
    cd_meta_map[key] = meta

cd_split_map = {}
cd_hub_map   = {}

for (pw, sid, destk), g in pickup_cd.groupby(["pickup_window", "step2_batch_id", "_dest_key"], dropna=False, sort=False):
    key = (str(pw), str(sid), str(destk))
    meta = cd_meta_map.get(key, {})
    kept_set = set(meta.get("coach_set", set()) or set())
    chain_text = str(meta.get("chain_order", "")).strip()
    hub = last_hotel_from_chain(chain_text)
    if hub is None:
        hub = meta.get("must_include", None)

    cd_hub_map[key] = hub

    hotels_list = (
        g[["From_clean"]].drop_duplicates(subset=["From_clean"])["From_clean"]
        .dropna().astype(str).tolist()
    )
    for h in hotels_list:
        cd_split_map[(str(pw), str(sid), str(destk), str(h))] = ("comfi_dest" if h in kept_set else "comfi")

df.loc[mask_pick_comfi_dest, "step2_vehicle_batch"] = df.loc[mask_pick_comfi_dest, ["pickup_window", "step2_batch_id", "_dest_key", "From_clean"]].apply(
    lambda r: cd_split_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"]), str(r["From_clean"])), "comfi_dest"),
    axis=1
)

df.loc[mask_pick_comfi_dest, "nearest_to_destination_hotel"] = df.loc[mask_pick_comfi_dest, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: cd_meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("nearest_hotel"),
    axis=1
)
df.loc[mask_pick_comfi_dest, "nearest_to_destination_min"] = df.loc[mask_pick_comfi_dest, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: cd_meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("nearest_min"),
    axis=1
)
df.loc[mask_pick_comfi_dest, "coach_anchor_used"] = df.loc[mask_pick_comfi_dest, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: cd_meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("anchor_used"),
    axis=1
)
df.loc[mask_pick_comfi_dest, "coach_chain_order"] = df.loc[mask_pick_comfi_dest, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: cd_meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("chain_order"),
    axis=1
)
df.loc[mask_pick_comfi_dest, "coach_chain_time_min"] = df.loc[mask_pick_comfi_dest, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: cd_meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("chain_time"),
    axis=1
)
df.loc[mask_pick_comfi_dest, "coach_must_include_hotel"] = df.loc[mask_pick_comfi_dest, ["pickup_window", "step2_batch_id", "_dest_key"]].apply(
    lambda r: cd_meta_map.get((str(r["pickup_window"]), str(r["step2_batch_id"]), str(r["_dest_key"])), {}).get("must_include"),
    axis=1
)

mask_return_step2 = df["trip_type"].str.lower().str.contains("return", na=False) & df["step2_batch_id"].notna()
df.loc[mask_return_step2, "step2_vehicle_batch"] = "coach"

# =====================================================
# STEP 3 (PICKUP)
# =====================================================
def _is_coach_like(row):
    s3 = str(row.get("step3_vehicle_batch", "")).strip().lower()
    if s3 in ("coach", "comfi_dest"):
        return True
    s2 = str(row.get("step2_vehicle_batch", "")).strip().lower()
    return s2 in ("coach", "comfi_dest")

def split_into_chunks(total_pax, chunk_size=13):
    total_pax = int(total_pax)
    out = []
    while total_pax > 0:
        take = min(chunk_size, total_pax)
        out.append(take)
        total_pax -= take
    return out

def route_time_minutes(path_hotels, drop_point):
    if not path_hotels or drop_point is None:
        return np.inf
    total = float(WAIT_MIN_PER_STOP) * len(path_hotels)

    for i in range(len(path_hotels) - 1):
        d = hotel_to_hotel_min(path_hotels[i], path_hotels[i+1])
        if np.isinf(d):
            return np.inf
        total += float(d)

    dlast = hotel_to_hotel_min(path_hotels[-1], drop_point)
    if np.isinf(dlast):
        return np.inf
    total += float(dlast)
    return float(total)

def feasible_alone(hotel, drop_point):
    t = float(WAIT_MIN_PER_STOP) + float(hotel_to_hotel_min(hotel, drop_point))
    if np.isinf(t):
        return False
    return t <= COACH_MAX_ROUTE_MIN + 1e-9

def build_route_seed_nearest_to_drop(tasks, drop_point):
    if not tasks:
        return [], [], np.inf, 0

    tasks_sorted = sorted(
        tasks,
        key=lambda x: (hotel_to_hotel_min(x["hotel"], drop_point), str(x["hotel"]), -int(x["pax"]))
    )
    seed = tasks_sorted[0]
    route_tasks = [seed]
    route_hotels = [seed["hotel"]]
    used_ids = {seed["task_id"]}
    total_pax = int(seed["pax"])
    total_time = route_time_minutes(route_hotels, drop_point)

    while True:
        remaining = [t for t in tasks if t["task_id"] not in used_ids]
        if not remaining:
            break

        last_hotel = route_hotels[-1]
        candidates = []
        for cand in remaining:
            cand_pax = int(cand["pax"])
            if total_pax + cand_pax > 13:
                continue

            trial_hotels = route_hotels + [cand["hotel"]]
            trial_time = route_time_minutes(trial_hotels, drop_point)
            if trial_time <= COACH_MAX_ROUTE_MIN + 1e-9:
                d = hotel_to_hotel_min(last_hotel, cand["hotel"])
                candidates.append((d, str(cand["hotel"]), -cand_pax, cand, trial_time))

        if not candidates:
            break

        candidates.sort()
        _d, _nm, _negp, chosen, chosen_time = candidates[0]
        route_tasks.append(chosen)
        route_hotels.append(chosen["hotel"])
        used_ids.add(chosen["task_id"])
        total_pax += int(chosen["pax"])
        total_time = float(chosen_time)

    return route_tasks, route_hotels, float(total_time), int(total_pax)

df["step3_vehicle_batch"] = None
df["step3_drop_point"] = None
df["step3_comfi_details"] = None
df["step3_comfi_vehicle_id"] = None
df["step3_comfi_vehicle_cap"] = None

step3_label_map = {}
step3_drop_map  = {}
step3_details_map = {}
step3_vid_map = {}
step3_vcap_map = {}

def _append_str(prev, add, sep=" + "):
    prev = "" if prev is None else str(prev)
    add = "" if add is None else str(add)
    if not prev:
        return add
    if not add:
        return prev
    return prev + sep + add

def _mark_required(group_key, hotel, required_pax, drop_point_label, drop_point_value):
    required_pax = int(required_pax)
    sug_txt, cap_str = suggest_vehicle_for_group(required_pax, "comfi")
    label = f"COMFI REQUIRED ({sug_txt}) [pax={required_pax}]"

    step3_label_map[(group_key, hotel)] = _append_str(step3_label_map.get((group_key, hotel)), label)
    step3_drop_map[(group_key, hotel)] = drop_point_label
    step3_vid_map[(group_key, hotel)]  = _append_str(step3_vid_map.get((group_key, hotel)), "")
    step3_vcap_map[(group_key, hotel)] = _append_str(step3_vcap_map.get((group_key, hotel)), cap_str)

    seq = f"{hotel} -> {drop_point_value}" if drop_point_value else hotel
    det = f"{label} | {drop_point_label} | {seq}"
    step3_details_map[(group_key, hotel)] = _append_str(step3_details_map.get((group_key, hotel)), det, sep=" || ")

mask_pick = mask_pick_all_step2

for pw, win_df in df[mask_pick].groupby("pickup_window", sort=False):
    comfi_pool = master_comfi_pool.copy()

    for sid, g in win_df.groupby("step2_batch_id", dropna=False, sort=False):
        group_key = (str(pw), str(sid))

        destination = (
            g["_dest_key"].dropna().astype(str).iloc[0]
            if g["_dest_key"].notna().any()
            else ""
        )

        chain_any = (
            g.loc[g["coach_chain_order"].astype(str).str.strip().ne(""), "coach_chain_order"]
             .astype(str).iloc[0]
            if (g["coach_chain_order"].astype(str).str.strip().ne("")).any()
            else ""
        )
        hub = last_hotel_from_chain(chain_any)
        if hub is None:
            if g["coach_must_include_hotel"].notna().any():
                hub = g["coach_must_include_hotel"].dropna().astype(str).iloc[0]
            elif g["nearest_to_destination_hotel"].notna().any():
                hub = g["nearest_to_destination_hotel"].dropna().astype(str).iloc[0]
            else:
                hub = None

        dk = g["_dest_key"].dropna().astype(str).iloc[0] if g["_dest_key"].notna().any() else ""
        hub = cd_hub_map.get((str(pw), str(sid), str(dk)), hub)

        hotels_unique = g[["From_clean", "step2_vehicle_batch"]].drop_duplicates(subset=["From_clean"]).copy()
        coach_hotels = set(hotels_unique.loc[
            hotels_unique["step2_vehicle_batch"].astype(str).str.lower().str.strip().isin(["coach", "comfi_dest"]),
            "From_clean"
        ].tolist())
        comfi_hotels = set(hotels_unique.loc[
            hotels_unique["step2_vehicle_batch"].astype(str).str.lower().str.strip() == "comfi", "From_clean"
        ].tolist())

        for h in coach_hotels:
            step3_label_map[(group_key, h)] = "coach" if (str(g.loc[g["From_clean"] == h, "step2_vehicle_batch"].iloc[0]).lower().strip() == "coach") else "comfi_dest"
            step3_drop_map[(group_key, h)] = ""
            step3_details_map[(group_key, h)] = ""
            step3_vid_map[(group_key, h)] = ""
            step3_vcap_map[(group_key, h)] = ""

        if (g["step2_vehicle_batch"].astype(str).str.lower().str.strip() == "comfi_dest").all():
            continue

        if not comfi_hotels:
            continue

        hotel_packs = g.groupby("From_clean", dropna=False)["total_pack"].sum().to_dict()

        tasks = []
        tid = 1
        for h in sorted(comfi_hotels):
            pax = int(hotel_packs.get(h, 0))
            if pax <= 0:
                continue
            for ch_pax in split_into_chunks(pax, 13):
                tasks.append({"task_id": f"{group_key[1]}_{tid:04d}", "hotel": h, "pax": int(ch_pax)})
                tid += 1

        if not tasks:
            continue

        routes_meta = []

        if hub is None:
            for t in tasks:
                h = t["hotel"]
                veh, comfi_pool_new = pick_best_fit_vehicle(comfi_pool, int(t["pax"]))
                if veh is None:
                    _mark_required(group_key, h, required_pax=int(t["pax"]),
                                   drop_point_label="destination", drop_point_value=destination)
                    continue

                comfi_pool = comfi_pool_new
                label = f"{veh['id']} ({veh['cap']} seater) - direct to destination"

                step3_label_map[(group_key, h)] = _append_str(step3_label_map.get((group_key, h)), label)
                step3_drop_map[(group_key, h)] = "destination"
                step3_vid_map[(group_key, h)]  = _append_str(step3_vid_map.get((group_key, h)), str(veh["id"]))
                step3_vcap_map[(group_key, h)] = _append_str(step3_vcap_map.get((group_key, h)), str(veh["cap"]))

                routes_meta.append({"label": label, "drop": "destination", "drop_point": destination,
                                    "time": np.nan, "hotels": [h]})

        else:
            hub_ok, hub_bad = [], []
            for t in tasks:
                if feasible_alone(t["hotel"], hub):
                    hub_ok.append(t)
                else:
                    hub_bad.append(t)

            while hub_ok:
                route_tasks, route_hotels, route_time, route_pax = build_route_seed_nearest_to_drop(hub_ok, hub)
                used_ids = {x["task_id"] for x in route_tasks}
                hub_ok = [x for x in hub_ok if x["task_id"] not in used_ids]

                veh, comfi_pool_new = pick_best_fit_vehicle(comfi_pool, route_pax)
                if veh is None:
                    for t in route_tasks:
                        _mark_required(group_key, t["hotel"], required_pax=int(t["pax"]),
                                       drop_point_label="hub", drop_point_value=hub)
                    continue

                comfi_pool = comfi_pool_new
                label = f"{veh['id']} ({veh['cap']} seater)"
                routes_meta.append({"label": label, "drop": "hub", "drop_point": hub,
                                    "time": route_time, "hotels": route_hotels})

                for t in route_tasks:
                    h = t["hotel"]
                    step3_label_map[(group_key, h)] = _append_str(step3_label_map.get((group_key, h)), label)
                    step3_drop_map[(group_key, h)] = "hub"
                    step3_vid_map[(group_key, h)]  = _append_str(step3_vid_map.get((group_key, h)), str(veh["id"]))
                    step3_vcap_map[(group_key, h)] = _append_str(step3_vcap_map.get((group_key, h)), str(veh["cap"]))

            direct_tasks, unreachable_direct = [], []
            for t in hub_bad:
                if feasible_alone(t["hotel"], destination):
                    direct_tasks.append(t)
                else:
                    unreachable_direct.append(t)

            while direct_tasks:
                route_tasks, route_hotels, route_time, route_pax = build_route_seed_nearest_to_drop(direct_tasks, destination)
                used_ids = {x["task_id"] for x in route_tasks}
                direct_tasks = [x for x in direct_tasks if x["task_id"] not in used_ids]

                veh, comfi_pool_new = pick_best_fit_vehicle(comfi_pool, route_pax)
                if veh is None:
                    for t in route_tasks:
                        _mark_required(group_key, t["hotel"], required_pax=int(t["pax"]),
                                       drop_point_label="destination", drop_point_value=destination)
                    continue

                comfi_pool = comfi_pool_new
                label = f"{veh['id']} ({veh['cap']} seater) - direct to destination"
                routes_meta.append({"label": label, "drop": "destination", "drop_point": destination,
                                    "time": route_time, "hotels": route_hotels})

                for t in route_tasks:
                    h = t["hotel"]
                    step3_label_map[(group_key, h)] = _append_str(step3_label_map.get((group_key, h)), label)
                    step3_drop_map[(group_key, h)] = "destination"
                    step3_vid_map[(group_key, h)]  = _append_str(step3_vid_map.get((group_key, h)), str(veh["id"]))
                    step3_vcap_map[(group_key, h)] = _append_str(step3_vcap_map.get((group_key, h)), str(veh["cap"]))

            for t in unreachable_direct:
                h = t["hotel"]
                veh, comfi_pool_new = pick_best_fit_vehicle(comfi_pool, int(t["pax"]))
                if veh is None:
                    _mark_required(group_key, h, required_pax=int(t["pax"]),
                                   drop_point_label="destination", drop_point_value=destination)
                    continue

                comfi_pool = comfi_pool_new
                label = f"{veh['id']} ({veh['cap']} seater) - direct to destination"

                step3_label_map[(group_key, h)] = _append_str(step3_label_map.get((group_key, h)), label)
                step3_drop_map[(group_key, h)] = "destination"
                step3_vid_map[(group_key, h)]  = _append_str(step3_vid_map.get((group_key, h)), str(veh["id"]))
                step3_vcap_map[(group_key, h)] = _append_str(step3_vcap_map.get((group_key, h)), str(veh["cap"]))

                routes_meta.append({"label": label, "drop": "destination", "drop_point": destination,
                                    "time": np.nan, "hotels": [h]})

        hotel_to_details = {h: [] for h in comfi_hotels}
        for rm in routes_meta:
            seq = rm["hotels"] + [rm["drop_point"]]
            seq_str = " -> ".join([x for x in seq if str(x).strip()])
            t_str = "" if (rm.get("time") is None or np.isnan(rm.get("time"))) else f"{rm['time']:.0f}min"
            line = f"{rm['label']} | {rm['drop']} | {t_str} | {seq_str}"
            for hh in rm["hotels"]:
                if hh in hotel_to_details:
                    hotel_to_details[hh].append(line)

        for h in comfi_hotels:
            existing = step3_details_map.get((group_key, h))
            add = " || ".join(hotel_to_details.get(h, []))
            step3_details_map[(group_key, h)] = _append_str(existing, add, sep=" || ")

df.loc[mask_pick, "step3_vehicle_batch"] = df.loc[mask_pick, ["pickup_window", "step2_batch_id", "From_clean"]].apply(
    lambda r: step3_label_map.get(((str(r["pickup_window"]), str(r["step2_batch_id"])), str(r["From_clean"])), None),
    axis=1
)
df.loc[mask_pick, "step3_drop_point"] = df.loc[mask_pick, ["pickup_window", "step2_batch_id", "From_clean"]].apply(
    lambda r: step3_drop_map.get(((str(r["pickup_window"]), str(r["step2_batch_id"])), str(r["From_clean"])), None),
    axis=1
)
df.loc[mask_pick, "step3_comfi_details"] = df.loc[mask_pick, ["pickup_window", "step2_batch_id", "From_clean"]].apply(
    lambda r: step3_details_map.get(((str(r["pickup_window"]), str(r["step2_batch_id"])), str(r["From_clean"])), None),
    axis=1
)
df.loc[mask_pick, "step3_comfi_vehicle_id"] = df.loc[mask_pick, ["pickup_window", "step2_batch_id", "From_clean"]].apply(
    lambda r: step3_vid_map.get(((str(r["pickup_window"]), str(r["step2_batch_id"])), str(r["From_clean"])), None),
    axis=1
)
df.loc[mask_pick, "step3_comfi_vehicle_cap"] = df.loc[mask_pick, ["pickup_window", "step2_batch_id", "From_clean"]].apply(
    lambda r: step3_vcap_map.get(((str(r["pickup_window"]), str(r["step2_batch_id"])), str(r["From_clean"])), None),
    axis=1
)

mask_cd_final = mask_pick_all_step2 & (df["step2_vehicle_batch"].astype(str).str.lower().str.strip() == "comfi_dest")
df.loc[mask_cd_final, "step3_vehicle_batch"] = "comfi_dest"
df.loc[mask_cd_final, "step3_drop_point"] = None
df.loc[mask_cd_final, "step3_comfi_details"] = None
df.loc[mask_cd_final, "step3_comfi_vehicle_id"] = None
df.loc[mask_cd_final, "step3_comfi_vehicle_cap"] = None

# =====================================================
# PICKUP ORDER
# =====================================================
df["pickup_order"] = None

def _split_vehicle_ids(x):
    s = "" if x is None else str(x).strip()
    if not s or s.lower() in ("nan", "none"):
        return []
    parts = [p.strip() for p in s.split("+")]
    return [p for p in parts if p]

def build_time_adj(nodes):
    nodes = set(nodes)
    adj = {n: {} for n in nodes}
    for (a, b), t in dur_direct.items():
        if a in nodes and b in nodes and not np.isinf(t):
            t = float(t)
            if b not in adj[a] or t < adj[a][b]:
                adj[a][b] = t
            if a not in adj[b] or t < adj[b][a]:
                adj[b][a] = t
    return adj

def dijkstra_time(adj, src):
    dist = {src: 0.0}
    pq = [(0.0, src)]
    seen = set()
    while pq:
        d, u = heapq.heappop(pq)
        if u in seen:
            continue
        seen.add(u)
        for v, w in adj.get(u, {}).items():
            nd = d + float(w)
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

def greedy_chain(nodes_list, start_node, adj):
    nodes = list(dict.fromkeys([n for n in nodes_list if pd.notna(n)]))
    if not nodes:
        return []
    if start_node not in nodes:
        start_node = sorted(nodes)[0]

    remaining = set(nodes)
    chain = []
    current = start_node

    while remaining:
        if current in remaining:
            chain.append(current)
            remaining.remove(current)
        if not remaining:
            break

        dist_map = dijkstra_time(adj, current)
        best, best_d = None, np.inf
        for cand in remaining:
            d = dist_map.get(cand, np.inf)
            if (d < best_d) or (d == best_d and str(cand) < str(best)):
                best_d = d
                best = cand
        if best is None or np.isinf(best_d):
            best = sorted(list(remaining))[0]
        current = best

    return chain

def _get_hub_from_group(g):
    chain_any = (
        g.loc[g["coach_chain_order"].astype(str).str.strip().ne(""), "coach_chain_order"]
         .astype(str).iloc[0]
        if (g["coach_chain_order"].astype(str).str.strip().ne("")).any()
        else ""
    )
    hub = last_hotel_from_chain(chain_any)
    if hub is None:
        if g["coach_must_include_hotel"].notna().any():
            hub = g["coach_must_include_hotel"].dropna().astype(str).iloc[0]
        elif g["nearest_to_destination_hotel"].notna().any():
            hub = g["nearest_to_destination_hotel"].dropna().astype(str).iloc[0]
    return hub

# 1) COACH-LIKE pickup_order (1..N) for coach + comfi_dest
for (pw, sid), g in df[mask_pick].groupby(["pickup_window", "step2_batch_id"], sort=False, dropna=False):
    hotels_unique = (
        g[["From_clean", "step3_vehicle_batch"]]
        .drop_duplicates(subset=["From_clean"])
        .copy()
    )
    coach_hotels = hotels_unique.loc[hotels_unique.apply(_is_coach_like, axis=1), "From_clean"].dropna().astype(str).tolist()
    if not coach_hotels:
        continue

    dest = g["_dest_key"].dropna().astype(str).iloc[0] if g["_dest_key"].notna().any() else None

    start_coach = (
        g.loc[g["nearest_to_destination_hotel"].notna(), "nearest_to_destination_hotel"].astype(str).iloc[0]
        if g["nearest_to_destination_hotel"].notna().any()
        else None
    )

    if (start_coach not in set(coach_hotels)) and dest:
        best_h, best_t = None, np.inf
        for h in coach_hotels:
            t = dest_to_hotel_min(dest, h)
            if (t < best_t) or (t == best_t and str(h) < str(best_h)):
                best_t, best_h = t, h
        start_coach = best_h if best_h else coach_hotels[0]

    nodes = set(coach_hotels)
    if dest:
        nodes.add(str(dest))
    adj = build_time_adj(nodes)

    chain = greedy_chain(coach_hotels, start_coach, adj)
    coach_order_map = {h: (i + 1) for i, h in enumerate(chain)}

    idx = g.index
    coach_mask = df.loc[idx].apply(_is_coach_like, axis=1)
    df.loc[idx[coach_mask], "pickup_order"] = df.loc[idx[coach_mask], "From_clean"].astype(str).map(coach_order_map)

# 2) COMFI pickup_order (per vehicle: 1..K)
df["_comfi_vids_list"] = df["step3_comfi_vehicle_id"].apply(_split_vehicle_ids)

for (pw, sid), g in df[mask_pick].groupby(["pickup_window", "step2_batch_id"], sort=False, dropna=False):
    comfi_rows = g[
        (~g.apply(_is_coach_like, axis=1)) &
        (g["_comfi_vids_list"].apply(lambda lst: len(lst) > 0))
    ].copy()

    if comfi_rows.empty:
        continue

    hub = _get_hub_from_group(g)

    exploded = []
    for ridx, row in comfi_rows.iterrows():
        hotel = str(row["From_clean"])
        for vid in row["_comfi_vids_list"]:
            exploded.append((ridx, hotel, str(vid)))
    exp_df = pd.DataFrame(exploded, columns=["row_index", "hotel", "vid"])

    all_comfi_hotels = sorted(exp_df["hotel"].unique().tolist())
    nodes = set(all_comfi_hotels)
    if hub:
        nodes.add(str(hub))
    adj = build_time_adj(nodes)

    for vid, sub in exp_df.groupby("vid", sort=False):
        hotels_vid = sorted(sub["hotel"].unique().tolist())
        if not hotels_vid:
            continue

        if hub and (str(hub) in adj):
            dist_from_hub = dijkstra_time(adj, str(hub))
            best_h, best_t = None, np.inf
            for h in hotels_vid:
                t = dist_from_hub.get(h, np.inf)
                if (t < best_t) or (t == best_t and str(h) < str(best_h)):
                    best_t, best_h = t, h
            start = best_h if best_h else hotels_vid[0]
        else:
            start = hotels_vid[0]

        chain = greedy_chain(hotels_vid, start, adj)
        comfi_order_map = {h: (i + 1) for i, h in enumerate(chain)}

        affected_rows = sub["row_index"].unique().tolist()
        df.loc[affected_rows, "pickup_order"] = df.loc[affected_rows, "From_clean"].astype(str).map(comfi_order_map)

df.drop(columns=["_comfi_vids_list"], inplace=True, errors="ignore")
df["pickup_order"] = pd.to_numeric(df["pickup_order"], errors="coerce").astype("Int64")

# =====================================================
# assigned_vehicle_batch
# =====================================================
df["assigned_vehicle_batch"] = None
mask_step3_coachlike = df["step3_vehicle_batch"].astype(str).str.strip().str.lower().isin(["coach", "comfi_dest"])
df.loc[mask_step3_coachlike, "assigned_vehicle_batch"] = df.loc[mask_step3_coachlike, "vehicle_label"]
df.loc[~mask_step3_coachlike, "assigned_vehicle_batch"] = df.loc[~mask_step3_coachlike, "step3_vehicle_batch"]

# =====================================================
# optimized_vehicle_id
# =====================================================
def _parse_window_to_minutes(pw_text):
    if pd.isna(pw_text):
        return (None, None)
    s = str(pw_text).strip()
    if "-" not in s:
        return (None, None)
    a, b = [x.strip() for x in s.split("-", 1)]
    try:
        ta = pd.to_datetime(a, format="%I:%M %p")
        tb = pd.to_datetime(b, format="%I:%M %p")
        sa = int(ta.hour) * 60 + int(ta.minute)
        sb = int(tb.hour) * 60 + int(tb.minute)
        return (sa, sb)
    except:
        return (None, None)

def _split_vids(x):
    s = "" if pd.isna(x) else str(x).strip()
    if not s or s.lower() in ("nan", "none"):
        return []
    return [p.strip() for p in s.split("+") if p.strip()]

def _is_direct_to_destination(step3_text):
    s = "" if pd.isna(step3_text) else str(step3_text).lower()
    return ("direct to destination" in s)

def _extract_cap_from_label(text):
    s = "" if pd.isna(text) else str(text)
    m = re.search(r"\((\d+)\s*seater\)", s, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

def _allocate_day_level(tasks, master_pool, required_label_fn=None):
    unused = [{"id": v["id"], "cap": int(v["cap"])} for v in master_pool]
    used_state = {}
    out = {}

    tasks_sorted = sorted(
        tasks,
        key=lambda x: (int(x["start"]), -int(x["req_cap"]), int(x["end"]), str(x["key"]))
    )

    for t in tasks_sorted:
        key = t["key"]
        start = int(t["start"])
        end = int(t["end"])
        req = int(t["req_cap"])
        buffer_min = int(t["buffer"])

        candidates = []
        for vid, st in used_state.items():
            if int(st["avail"]) <= start and int(st["cap"]) >= req:
                candidates.append((int(st["cap"]), str(vid), vid))
        candidates.sort()

        if candidates:
            _cap, _nm, chosen_vid = candidates[0]
            chosen_cap = int(used_state[chosen_vid]["cap"])
        else:
            chosen_vid = None
            chosen_cap = None
            for i, v in enumerate(unused):
                if int(v["cap"]) >= req:
                    chosen_vid = v["id"]
                    chosen_cap = int(v["cap"])
                    unused.pop(i)
                    break

            if chosen_vid is None:
                out[key] = required_label_fn(req) if required_label_fn else f"REQUIRED ({req})"
                continue

        out[key] = chosen_vid
        used_state[chosen_vid] = {"cap": chosen_cap, "avail": end + buffer_min}

    return out

df["optimized_vehicle_id"] = None
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["_pw_start_min"], df["_pw_end_min"] = zip(*df["pickup_window"].apply(_parse_window_to_minutes))

# ✅ NEW: compute per-day per-window "early start -10" only when gap == 30
def build_day_pw_adjstart_map(df_in):
    out = {}
    base = df_in[df_in["_pw_start_min"].notna() & df_in["_pw_end_min"].notna() & df_in["Date"].notna()].copy()
    for day, ddf in base.groupby(df_in["Date"].dt.date, sort=True):
        pw_tbl = (
            ddf[["pickup_window", "_pw_start_min", "_pw_end_min"]]
            .drop_duplicates()
            .sort_values("_pw_start_min")
            .to_dict("records")
        )
        for i, rec in enumerate(pw_tbl):
            pw = str(rec["pickup_window"])
            start_min = int(rec["_pw_start_min"])
            end_min = int(rec["_pw_end_min"])
            adj_start = start_min

            if i < len(pw_tbl) - 1:
                nxt = pw_tbl[i + 1]
                nxt_start = int(nxt["_pw_start_min"])
                gap = nxt_start - end_min
                if gap == 30:
                    adj_start = start_min - 10   # ✅ allow 10 min early

            out[(day, pw)] = int(adj_start)
    return out

day_pw_adjstart = build_day_pw_adjstart_map(df)

mask_day_opt = (
    df["trip_type"].astype(str).str.lower().str.contains("pickup|return", na=False)
    & df["step2_batch_id"].notna()
    & df["_pw_start_min"].notna()
    & df["_pw_end_min"].notna()
    & df["Date"].notna()
)

for day, day_df in df[mask_day_opt].groupby(df["Date"].dt.date, sort=True):

    # =====================================================
    # 1) COACH tasks (pickup coach + return)
    #    ✅ UPDATED:
    #      - if this pickup_window has exact 30-min gap to next window,
    #        task start can be 10-min early (start_min - 10)
    #      - this enables same coach reuse as usual (buffer logic unchanged)
    # =====================================================
    coach_rows = day_df[
        day_df["step2_batch_id"].notna()
        & (
            day_df["trip_type"].astype(str).str.lower().str.contains("return", na=False)
            | day_df["step3_vehicle_batch"].astype(str).str.strip().str.lower().eq("coach")
        )
    ].copy()

    if not coach_rows.empty:
        coach_tasks = []
        for (pw, sid), g in coach_rows.groupby(["pickup_window", "step2_batch_id"], dropna=False, sort=False):
            pw = str(pw)
            start_min = int(g["_pw_start_min"].iloc[0])
            end_min   = int(g["_pw_end_min"].iloc[0])

            is_return = g["trip_type"].astype(str).str.lower().str.contains("return", na=False).any()

            # ✅ only pickup coach can start 10 min early (when eligible)
            if (not is_return):
                start_min = int(day_pw_adjstart.get((day, pw), start_min))

            req_cap = None
            if g["vehicle_capacity"].notna().any():
                req_cap = int(g["vehicle_capacity"].dropna().astype(int).iloc[0])
            if req_cap is None:
                req_cap = int(g["total_pack"].fillna(0).sum())

            coach_tasks.append({
                "key": (str(pw), str(sid)),
                "start": int(start_min),
                "end": int(end_min),
                "req_cap": int(req_cap),
                "buffer": 60 if is_return else 30
            })

        coach_opt_map = _allocate_day_level(
            coach_tasks,
            master_coach_pool,
            required_label_fn=lambda req: f"COACH REQUIRED ({suggest_vehicle_for_group(int(req),'coach')[0]}) [pax={int(req)}]"
        )

        df.loc[coach_rows.index, "optimized_vehicle_id"] = coach_rows.apply(
            lambda r: coach_opt_map.get((str(r["pickup_window"]), str(r["step2_batch_id"])), None),
            axis=1
        )

    # =====================================================
    # 2) ALL COMFI tasks together (normal comfi + comfi_dest)  ✅ keep
    # =====================================================
    comfi_rows = day_df[
        (~day_df.apply(_is_coach_like, axis=1))
        & day_df["step3_comfi_vehicle_id"].notna()
    ].copy()

    exp = []

    if not comfi_rows.empty:
        comfi_rows["_vids"] = comfi_rows["step3_comfi_vehicle_id"].apply(_split_vids)

        for idx, r in comfi_rows.iterrows():
            pw = str(r["pickup_window"])
            start_min = int(r["_pw_start_min"])
            end_min   = int(r["_pw_end_min"])

            step3_txt = r.get("step3_vehicle_batch", "")
            is_direct = _is_direct_to_destination(step3_txt)
            cap_guess = _extract_cap_from_label(step3_txt)

            for vid in r["_vids"]:
                exp.append({
                    "row_index": int(idx),
                    "kind": "comfi",
                    "pw": pw,
                    "start": start_min,
                    "end": end_min,
                    "tmp_vid": str(vid),
                    "req_cap": int(cap_guess) if cap_guess is not None else 13,
                    "buffer": 30 if bool(is_direct) else 0
                })

    cd_rows = day_df[
        (day_df["step3_vehicle_batch"].astype(str).str.strip().str.lower() == "comfi_dest")
        & day_df["step2_batch_id"].notna()
    ].copy()

    if not cd_rows.empty:
        for (pw, sid), g in cd_rows.groupby(["pickup_window", "step2_batch_id"], dropna=False, sort=False):
            start_min = int(g["_pw_start_min"].iloc[0])
            end_min   = int(g["_pw_end_min"].iloc[0])

            req_cap = int(g["total_pack"].fillna(0).sum())  # pax of this step2 group

            tmp_vid = f"CD::{sid}"  # stable temp key per step2 batch

            exp.append({
                "row_index": None,
                "kind": "comfi_dest",
                "pw": str(pw),
                "start": start_min,
                "end": end_min,
                "tmp_vid": tmp_vid,
                "req_cap": int(req_cap),
                "buffer": 30
            })

    if exp:
        exp_df = pd.DataFrame(exp)

        comfi_tasks = []
        for (pw, tmp_vid), sub in exp_df.groupby(["pw", "tmp_vid"], sort=False):
            start_min = int(sub["start"].iloc[0])
            end_min   = int(sub["end"].iloc[0])

            req_cap = int(sub["req_cap"].max())
            buffer_min = int(sub["buffer"].max())

            comfi_tasks.append({
                "key": (str(pw), str(tmp_vid)),
                "start": int(start_min),
                "end": int(end_min),
                "req_cap": int(req_cap),
                "buffer": int(buffer_min)
            })

        comfi_opt_map = _allocate_day_level(
            comfi_tasks,
            master_comfi_pool,
            required_label_fn=lambda req: f"COMFI REQUIRED ({suggest_vehicle_for_group(int(req),'comfi')[0]}) [pax={int(req)}]"
        )

        if not comfi_rows.empty:
            def _map_comfi_row_to_opt(pw, vids):
                mapped = [comfi_opt_map.get((str(pw), str(v)), None) for v in vids]
                if (not mapped) or any(x is None for x in mapped):
                    return None
                return " + ".join(mapped)

            df.loc[comfi_rows.index, "optimized_vehicle_id"] = comfi_rows.apply(
                lambda r: _map_comfi_row_to_opt(r["pickup_window"], _split_vids(r["step3_comfi_vehicle_id"])),
                axis=1
            )

        df.loc[cd_rows.index, "optimized_vehicle_id"] = cd_rows.apply(
    lambda r: comfi_opt_map.get((str(r["pickup_window"]), f"CD::{str(r['step2_batch_id'])}"), None),
    axis=1
)


    mask_required_day = day_df["step3_vehicle_batch"].astype(str).str.upper().str.contains("COMFI REQUIRED", na=False)
    if mask_required_day.any():
        idxs = day_df.index[mask_required_day]
        df.loc[idxs, "optimized_vehicle_id"] = df.loc[idxs, "step3_vehicle_batch"]

df.drop(columns=["_pw_start_min", "_pw_end_min"], inplace=True, errors="ignore")

# =====================================================
# ✅ SHOW COMFI_DEST ALLOCATED VEHICLE IN assigned_vehicle_batch
#   (so comfi_dest rows don't keep old coach label / blank)
# =====================================================
# =====================================================
# ✅ ADD: assigned COMFI capacity (from optimized_vehicle_id)
# =====================================================
df["optimized_vehicle_capacity"] = None          # per-vehicle caps like "13+6"
df["optimized_vehicle_capacity_sum"] = None      # numeric total cap

def _split_plus_safe2(x):
    s = "" if pd.isna(x) else str(x).strip()
    if not s or s.lower() in ("nan", "none"):
        return []
    if "REQUIRED" in s.upper():
        return []
    return [p.strip() for p in s.split("+") if p.strip()]

def _caps_text_and_sum(vid_text):
    vids = _split_plus_safe2(vid_text)
    if not vids:
        return (None, None)

    caps = []
    total = 0
    for vid in vids:
        cap = vehicle_cap_lookup.get(str(vid))
        if cap is None:
            caps.append("")
        else:
            caps.append(str(int(cap)))
            total += int(cap)

    caps_txt = "+".join([c for c in caps if c != ""])
    if not caps_txt:
        caps_txt = None

    return (caps_txt, int(total) if total > 0 else None)

mask_has_opt = df["optimized_vehicle_id"].notna()

tmp = df.loc[mask_has_opt, "optimized_vehicle_id"].apply(_caps_text_and_sum)
df.loc[mask_has_opt, "optimized_vehicle_capacity"] = tmp.apply(lambda x: x[0])
df.loc[mask_has_opt, "optimized_vehicle_capacity_sum"] = tmp.apply(lambda x: x[1])

# ✅ For comfi_dest rows, show allocated COMFI id + its capacity in assigned columns
mask_cd_show = (
    df["step3_vehicle_batch"].astype(str).str.strip().str.lower().eq("comfi_dest")
    & df["optimized_vehicle_id"].notna()
)

df.loc[mask_cd_show, "assigned_vehicle_batch"] = df.loc[mask_cd_show, "optimized_vehicle_id"].astype(str)
df.loc[mask_cd_show, "assigned_vehicle_capacity"] = df.loc[mask_cd_show, "optimized_vehicle_capacity_sum"]



# =====================================================
# pickup_time
#   ✅ UPDATED (your rule):
#     If (end_of_this_window) to (start_of_next_window) == 30 min,
#     THEN this window may start 10 mins early.
# =====================================================
df["pickup_time"] = None

def _fmt_time(dt):
    return dt.strftime("%I:%M %p")

def _safe_travel_min(a, b):
    t = hotel_to_hotel_min(str(a), str(b))
    if np.isinf(t) or pd.isna(t):
        return 0.0
    return float(t)

def _split_opt_vids(x):
    s = "" if pd.isna(x) else str(x).strip()
    if not s or s.lower() in ("nan", "none"):
        return []
    return [p.strip() for p in s.split("+") if p.strip()]

def _is_required_vid(x):
    s = "" if pd.isna(x) else str(x).upper()
    return ("REQUIRED" in s)

df["_pw_start_min"], df["_pw_end_min"] = zip(*df["pickup_window"].apply(_parse_window_to_minutes))

# rebuild day_pw_adjstart for pickup_time using current mins
day_pw_adjstart = build_day_pw_adjstart_map(df)

coach_time_map = {}
coach_hub_arrival_map = {}
comfi_time_map = {}

mask_pick2 = (
    df["trip_type"].str.lower().str.contains("pickup", na=False)
    & df["step2_batch_id"].notna()
    & df["_pw_start_min"].notna()
    & df["_pw_end_min"].notna()
    & df["Date"].notna()
)

mask_coachlike_opt = (
    mask_pick2
    & df.apply(_is_coach_like, axis=1)
    & df["optimized_vehicle_id"].notna()
    & (~df["optimized_vehicle_id"].apply(_is_required_vid))
)

for (day, pw, sid, vid), g in df[mask_coachlike_opt].groupby(
    [df["Date"].dt.date, "pickup_window", "step2_batch_id", "optimized_vehicle_id"],
    sort=False,
    dropna=False
):
    pw = str(pw)
    pw_start_min = int(g["_pw_start_min"].iloc[0])

    # ✅ HERE is the real change: do NOT clamp; allow early if eligible
    adj_start_min = int(day_pw_adjstart.get((day, pw), pw_start_min))

    day0 = pd.Timestamp(day)
    cur_time = day0 + pd.Timedelta(minutes=int(adj_start_min))

    hotels_df = (
        g[["From_clean", "pickup_order"]]
        .drop_duplicates(subset=["From_clean"])
        .copy()
    )
    hotels_df["pickup_order"] = pd.to_numeric(hotels_df["pickup_order"], errors="coerce")
    hotels_df = hotels_df.dropna(subset=["pickup_order"])
    hotels_df = hotels_df.sort_values("pickup_order", ascending=True)

    hotels = hotels_df["From_clean"].astype(str).tolist()
    if not hotels:
        continue

    hotel_time = {}
    hotel_time[hotels[0]] = cur_time

    for i in range(1, len(hotels)):
        prev_h = hotels[i - 1]
        next_h = hotels[i]
        travel = _safe_travel_min(prev_h, next_h)
        cur_time = cur_time + pd.Timedelta(minutes=float(WAIT_MIN_PER_STOP) + travel)
        hotel_time[next_h] = cur_time

    hub = _get_hub_from_group(g)
    if hub is not None and str(hub) in hotel_time:
        coach_hub_arrival_map[(day, str(pw), str(sid))] = hotel_time[str(hub)]

    for idx, r in g.iterrows():
        h = str(r["From_clean"])
        if h in hotel_time:
            coach_time_map[idx] = _fmt_time(hotel_time[h])

mask_comfi_opt = (
    mask_pick2
    & (~df.apply(_is_coach_like, axis=1))
    & df["optimized_vehicle_id"].notna()
    & (~df["optimized_vehicle_id"].apply(_is_required_vid))
)

exp_flat = []
for idx, r in df[mask_comfi_opt].iterrows():
    vids = _split_opt_vids(r["optimized_vehicle_id"])
    if not vids:
        continue
    for vid in vids:
        exp_flat.append({
            "row_idx": int(idx),
            "day": df.at[idx, "Date"].date() if pd.notna(df.at[idx, "Date"]) else None,
            "pickup_window": str(r["pickup_window"]),
            "sid": str(r["step2_batch_id"]),
            "hotel": str(r["From_clean"]),
            "vid": str(vid)
        })

exp_df = pd.DataFrame(exp_flat)
exp_df = exp_df.dropna(subset=["day"]) if not exp_df.empty else exp_df

hub_map = {}
for (day, pw, sid), gg in df[mask_pick2].groupby([df["Date"].dt.date, "pickup_window", "step2_batch_id"], sort=False):
    hub_map[(day, str(pw), str(sid))] = _get_hub_from_group(gg)

def _safe_pickup_order(day, pw, sid, hotel):
    sub = df.loc[
        (df["Date"].dt.date == day)
        & (df["pickup_window"].astype(str) == str(pw))
        & (df["step2_batch_id"].astype(str) == str(sid))
        & (df["From_clean"].astype(str) == str(hotel)),
        "pickup_order"
    ]
    sub = pd.to_numeric(sub, errors="coerce").dropna()
    return float(sub.iloc[0]) if len(sub) else np.nan

if not exp_df.empty:
    for (day, pw, sid, vid), sub in exp_df.groupby(["day", "pickup_window", "sid", "vid"], sort=False):
        hub = hub_map.get((day, str(pw), str(sid)), None)

        any_row = df[(df["Date"].dt.date == day) & (df["pickup_window"].astype(str) == str(pw))].head(1)
        if any_row.empty:
            continue
        pw_end_min = int(any_row["_pw_end_min"].iloc[0])
        day0 = pd.Timestamp(day)
        hub_time = day0 + pd.Timedelta(minutes=pw_end_min)

        coach_hub_t = coach_hub_arrival_map.get((day, str(pw), str(sid)), None)
        if coach_hub_t is not None and pd.notna(coach_hub_t):
            hub_time = pd.Timestamp(coach_hub_t)

        hotels_df = sub[["hotel"]].drop_duplicates().copy()
        hotels_df["pickup_order"] = hotels_df["hotel"].map(lambda h: _safe_pickup_order(day, pw, sid, h))
        hotels_df["pickup_order"] = pd.to_numeric(hotels_df["pickup_order"], errors="coerce")
        hotels_df = hotels_df.dropna(subset=["pickup_order"]).sort_values("pickup_order", ascending=True)

        hotels = hotels_df["hotel"].astype(str).tolist()
        if not hotels:
            continue

        hotel_time = {}
        last_h = hotels[-1]
        travel_last = _safe_travel_min(last_h, hub) if hub else 0.0
        t_last = hub_time - pd.Timedelta(minutes=float(WAIT_MIN_PER_STOP) + travel_last)
        hotel_time[last_h] = t_last

        for i in range(len(hotels) - 2, -1, -1):
            prev_h = hotels[i]
            next_h = hotels[i + 1]
            travel = _safe_travel_min(prev_h, next_h)
            t_prev = hotel_time[next_h] - pd.Timedelta(minutes=float(WAIT_MIN_PER_STOP) + travel)
            hotel_time[prev_h] = t_prev

        for row_idx, hotel in sub[["row_idx", "hotel"]].drop_duplicates().itertuples(index=False, name=None):
            if hotel not in hotel_time:
                continue
            tstr = _fmt_time(hotel_time[hotel])
            prev = comfi_time_map.get(int(row_idx), "")
            comfi_time_map[int(row_idx)] = (tstr if not prev else (prev + " + " + tstr))

if coach_time_map:
    df.loc[list(coach_time_map.keys()), "pickup_time"] = [coach_time_map[i] for i in coach_time_map.keys()]

if comfi_time_map:
    for idx, val in comfi_time_map.items():
        if pd.isna(df.at[idx, "pickup_time"]) or str(df.at[idx, "pickup_time"]).strip() == "":
            df.at[idx, "pickup_time"] = val

df.drop(columns=["_pw_start_min", "_pw_end_min"], inplace=True, errors="ignore")

# =====================================================
# FINAL: RETURN TRIP ENFORCEMENTS
# =====================================================
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df.loc[mask_return_step2, "pickup_order"] = pd.NA

def _window_start_str(pw):
    if pd.isna(pw):
        return None
    s = str(pw)
    if "-" not in s:
        return s.strip()
    return s.split("-", 1)[0].strip()

df.loc[mask_return_step2, "pickup_time"] = df.loc[mask_return_step2, "pickup_window"].apply(_window_start_str)

for col in ["nearest_to_destination_hotel", "nearest_to_destination_min", "coach_anchor_used",
            "coach_chain_order", "coach_chain_time_min", "coach_must_include_hotel",
            "step3_vehicle_batch", "step3_drop_point", "step3_comfi_details", "step3_comfi_vehicle_id", "step3_comfi_vehicle_cap",
            "assigned_vehicle_batch"]:
    if col in df.columns:
        df.loc[mask_return_step2, col] = None

# =====================================================
# FINAL OUTPUT SORT
# =====================================================
final_df = (
    df.sort_values(
        ["Date", "pickup_window", "assigned_vehicle_batch", "pickup_order", "batch_id", "step2_batch_id"],
        ascending=[True, True, True, True, True, True],
        na_position="last"
    )
    .reset_index(drop=True)
)

# =====================================================
# REPORTING SHEETS
# =====================================================
def _norm(x):
    s = "" if pd.isna(x) else str(x).strip()
    return s

def _split_plus(x):
    s = _norm(x)
    if not s or s.lower() in ("nan", "none"):
        return []
    return [p.strip() for p in s.split("+") if p.strip()]

def _pw_to_start_min(pw):
    if pd.isna(pw):
        return np.nan
    s = str(pw).strip()
    if "-" not in s:
        return np.nan
    a = s.split("-", 1)[0].strip()
    try:
        t = pd.to_datetime(a, format="%I:%M %p")
        return int(t.hour) * 60 + int(t.minute)
    except:
        return np.nan

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["_pw_start_min"] = df["pickup_window"].apply(_pw_to_start_min)

mask_valid_batch = df["step2_batch_id"].notna()

def _agg_hotels(g):
    sub = g[["From_clean", "pickup_order", "total_pack"]].drop_duplicates()
    sub["pickup_order"] = pd.to_numeric(sub["pickup_order"], errors="coerce")
    sub = sub.sort_values(["pickup_order", "From_clean"], na_position="last")
    return " | ".join([f"{r.From_clean}({int(r.total_pack)})" for r in sub.itertuples(index=False)])

batch_summary = (
    df[mask_valid_batch]
    .groupby(["Date", "trip_type", "pickup_window", "step2_batch_id"], dropna=False, sort=True)
    .apply(lambda g: pd.Series({
        "dest": _norm(g["To_clean"].dropna().astype(str).iloc[0]) if g["To_clean"].notna().any() else "",
        "batch_pax_total": int(g["total_pack"].fillna(0).sum()),
        "assigned_vehicle_batch": _norm(g["assigned_vehicle_batch"].dropna().astype(str).iloc[0]) if g["assigned_vehicle_batch"].notna().any() else "",
        "step2_vehicle_label": _norm(g["vehicle_label"].dropna().astype(str).iloc[0]) if g["vehicle_label"].notna().any() else "",
        "step3_batch_label_sample": _norm(g["step3_vehicle_batch"].dropna().astype(str).iloc[0]) if g["step3_vehicle_batch"].notna().any() else "",
        "step3_comfi_vehicle_ids": " + ".join(sorted(set(sum([_split_plus(x) for x in g["step3_comfi_vehicle_id"].dropna().astype(str)], [])))),
        "optimized_vehicle_id": " + ".join(sorted(set(sum([_split_plus(x) for x in g["optimized_vehicle_id"].dropna().astype(str)], [])))),
        "hotel_sequence_by_pickup_order": _agg_hotels(g),
        "pickup_time_min": _norm(g["pickup_time"].dropna().astype(str).min()) if g["pickup_time"].notna().any() else "",
        "pickup_time_max": _norm(g["pickup_time"].dropna().astype(str).max()) if g["pickup_time"].notna().any() else "",
    }))
    .reset_index()
)

# =====================================================
# Vehicle_Tasks + Vehicle_Timeline  (INCLUDE PICKUP + RETURN)
# =====================================================
tasks_rows = []

def _hotel_for_task_row(r):
    trip = str(r.get("trip_type", "")).lower()
    if "return" in trip:
        return str(r.get("Return_hotel_clean", r.get("To_clean", r.get("To", ""))))
    return str(r.get("From_clean", ""))

def _order_key_for_task_row(r):
    trip = str(r.get("trip_type", "")).lower()
    if "pickup" in trip:
        po = r.get("pickup_order", np.nan)
        try:
            return (0, float(po)) if pd.notna(po) else (0, 1e9)
        except:
            return (0, 1e9)
    pax = r.get("total_pack", 0)
    try:
        pax = int(pax) if pd.notna(pax) else 0
    except:
        pax = 0
    return (1, -pax)

def _split_plus(x):
    s = "" if pd.isna(x) else str(x).strip()
    if not s or s.lower() in ("nan", "none"):
        return []
    return [p.strip() for p in s.split("+") if p.strip()]

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["_pw_start_min"] = df["pickup_window"].apply(_pw_to_start_min)

mask_tasks = (
    df["trip_type"].astype(str).str.lower().str.contains("pickup|return", na=False)
    & df["step2_batch_id"].notna()
    & df["optimized_vehicle_id"].notna()
)

for (day, pw), g_pw in df[mask_tasks].groupby([df["Date"].dt.date, "pickup_window"], sort=True):
    pw_start = float(g_pw["_pw_start_min"].dropna().iloc[0]) if g_pw["_pw_start_min"].notna().any() else np.nan

    g_pw2 = g_pw.copy()
    g_pw2["_opt_vids"] = g_pw2["optimized_vehicle_id"].apply(_split_plus)
    g_pw2["_task_hotel"] = g_pw2.apply(_hotel_for_task_row, axis=1)

    exp = []
    for idx, r in g_pw2.iterrows():
        vids = r["_opt_vids"]
        if not vids:
            continue

        hotel = str(r["_task_hotel"])
        pax = int(r["total_pack"]) if pd.notna(r["total_pack"]) else 0
        ptime = "" if pd.isna(r.get("pickup_time")) else str(r.get("pickup_time")).strip()

        for vid in vids:
            exp.append({
                "Date": pd.Timestamp(day),
                "trip_type": str(r.get("trip_type", "")),
                "pickup_window": str(pw),
                "pw_start_min": pw_start,
                "step2_batch_id": str(r["step2_batch_id"]),
                "hotel": hotel,
                "pax": pax,
                "pickup_order": r.get("pickup_order", np.nan),
                "optimized_vid": str(vid),
                "pickup_time": ptime,
            })

    exp_df2 = pd.DataFrame(exp)
    if exp_df2.empty:
        continue

    for vid, sub in exp_df2.groupby("optimized_vid", sort=False):

        total_pax = int(sub["pax"].sum())
        hotels_unique = sub.drop_duplicates(subset=["hotel"]).copy()
        hotels_unique["_ok"] = hotels_unique.apply(_order_key_for_task_row, axis=1)
        hotels_unique = hotels_unique.sort_values(["_ok", "hotel"], na_position="last")
        hotels_str = " -> ".join([f"{x.hotel}({int(x.pax)})" for x in hotels_unique.itertuples(index=False)])

        any_time = ""
        if sub["pickup_time"].notna().any():
            any_time = str(sub["pickup_time"].dropna().astype(str).min())

        tasks_rows.append({
            "Date": pd.Timestamp(day),
            "trip_type": "pickup+return",
            "pickup_window": str(pw),
            "pw_start_min": pw_start,
            "optimized_vehicle_id": vid,
            "task_total_pax": total_pax,
            "hotels_in_order": hotels_str,
            "any_pickup_time": any_time,
        })

vehicle_tasks = pd.DataFrame(tasks_rows)
if not vehicle_tasks.empty:
    vehicle_tasks = vehicle_tasks.sort_values(["Date", "pw_start_min", "optimized_vehicle_id"], na_position="last")

timeline_rows = []
if not vehicle_tasks.empty:
    for day, ddf in vehicle_tasks.groupby(vehicle_tasks["Date"].dt.date, sort=True):
        first_use = (
            ddf.groupby("optimized_vehicle_id", sort=False)["pw_start_min"]
               .min()
               .reset_index()
               .sort_values(["pw_start_min", "optimized_vehicle_id"], na_position="last")
        )

        for seq, r in enumerate(first_use.itertuples(index=False), start=1):
            vid = r.optimized_vehicle_id
            uses = ddf[ddf["optimized_vehicle_id"] == vid].sort_values("pw_start_min")
            pw_list = " | ".join(uses["pickup_window"].astype(str).tolist())

            timeline_rows.append({
                "Date": pd.Timestamp(day),
                "reuse_sequence": seq,
                "optimized_vehicle_id": vid,
                "pickup_windows_used": pw_list,
                "vehicle_capacity": vehicle_cap_lookup.get(str(vid), np.nan),
                "num_tasks": int(len(uses)),
            })

vehicle_timeline = pd.DataFrame(timeline_rows)
if not vehicle_timeline.empty:
    vehicle_timeline = vehicle_timeline.sort_values(["Date", "reuse_sequence"], na_position="last")

# =====================================================
# UTILIZATION REPORT
# =====================================================
def _split_plus_safe(x):
    s = "" if pd.isna(x) else str(x).strip()
    if not s or s.lower() in ("nan", "none"):
        return []
    return [p.strip() for p in s.split("+") if p.strip()]

coach_ids = [str(v["id"]) for v in master_coach_pool]
comfi_ids = [str(v["id"]) for v in master_comfi_pool]

coach_cap = {str(v["id"]): int(v["cap"]) for v in master_coach_pool}
comfi_cap = {str(v["id"]): int(v["cap"]) for v in master_comfi_pool}

used_ids = set()
if "optimized_vehicle_id" in df.columns:
    for x in df["optimized_vehicle_id"].dropna().astype(str):
        if "REQUIRED" in x.upper():
            continue
        for vid in _split_plus_safe(x):
            used_ids.add(str(vid))

tasks_count_map = {}
windows_used_map = {}

if "vehicle_timeline" in locals() and not vehicle_timeline.empty:
    vt = vehicle_timeline.copy()
    vt["optimized_vehicle_id"] = vt["optimized_vehicle_id"].astype(str)
    tasks_count_map = vt.groupby("optimized_vehicle_id")["num_tasks"].sum().to_dict()
    windows_used_map = vt.groupby("optimized_vehicle_id")["pickup_windows_used"].apply(
        lambda s: " | ".join(sorted(set([w.strip() for w in s.dropna().astype(str).tolist() if w.strip()])))
    ).to_dict()
else:
    if "vehicle_tasks" in locals() and not vehicle_tasks.empty:
        vt2 = vehicle_tasks.copy()
        vt2["optimized_vehicle_id"] = vt2["optimized_vehicle_id"].astype(str)
        tasks_count_map = vt2.groupby("optimized_vehicle_id").size().to_dict()
        windows_used_map = vt2.groupby("optimized_vehicle_id")["pickup_window"].apply(
            lambda s: " | ".join(sorted(set(s.dropna().astype(str).tolist())))
        ).to_dict()

def _util_row(vid, vtype):
    cap = coach_cap.get(vid) if vtype == "coach" else comfi_cap.get(vid)
    used = "YES" if vid in used_ids else "NO"
    return {
        "vehicle_type": vtype,
        "vehicle_id": vid,
        "vehicle_capacity": cap,
        "used_in_plan": used,
        "num_tasks": int(tasks_count_map.get(vid, 0)),
        "pickup_windows_used": windows_used_map.get(vid, "")
    }

util_rows = []
for vid in coach_ids:
    util_rows.append(_util_row(vid, "coach"))
for vid in comfi_ids:
    util_rows.append(_util_row(vid, "comfi"))

vehicle_utilization = pd.DataFrame(util_rows)

unused_rows = vehicle_utilization[vehicle_utilization["used_in_plan"] == "NO"].copy()
unused_rows = unused_rows.sort_values(["vehicle_type", "vehicle_capacity", "vehicle_id"], na_position="last")

vehicle_utilization = vehicle_utilization.sort_values(
    ["vehicle_type", "used_in_plan", "vehicle_capacity", "vehicle_id"],
    ascending=[True, True, True, True],
    na_position="last"
).reset_index(drop=True)

unused_vehicles = unused_rows.reset_index(drop=True)

# helper column: remove before output
df.drop(columns=["_dest_key"], inplace=True, errors="ignore")
final_df.drop(columns=["_dest_key"], inplace=True, errors="ignore")

out_file = "89_with_vehicle_reports.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    final_df.to_excel(writer, index=False, sheet_name="Final_Output")
    batch_summary.to_excel(writer, index=False, sheet_name="Batch_Summary")
    vehicle_tasks.to_excel(writer, index=False, sheet_name="Vehicle_Tasks")
    vehicle_timeline.to_excel(writer, index=False, sheet_name="Vehicle_Timeline")
    vehicle_utilization.to_excel(writer, index=False, sheet_name="Vehicle_Utilization")
    unused_vehicles.to_excel(writer, index=False, sheet_name="Unused_Vehicles")

print(f"\n🎯 FINAL FILE GENERATED → {out_file}")
print("✔️ Sheets: Final_Output, Batch_Summary, Vehicle_Tasks, Vehicle_Timeline")

final_df.to_excel("89.xlsx", index=False)
print("🎯 FINAL FILE GENERATED → 89.xlsx")

C:\Users\Sakthi\AppData\Local\Temp\ipykernel_13876\562398868.py:83: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_batch_id)



Enter available coaches like:
Example => 2x23, 1x45, 1x49, 1x55

Coach pool template (per pickup window): ['coach001(23)', 'coach002(23)', 'coach003(45)', 'coach004(49)', 'coach005(55)']

Enter available COMFI like:
Example => 6x13, 2x6, 3x4

COMFI pool template (per pickup window): ['comfi006(4)', 'comfi007(4)', 'comfi008(4)', 'comfi004(6)', 'comfi005(6)', 'comfi001(13)', 'comfi002(13)', 'comfi003(13)']

[CAPACITY CHECK] pickup_window='01:45 PM - 02:15 PM' coach_pax=0 vs total_coach_capacity=195

[STEP2 COMFI_DEST ONLY] pickup_window='01:45 PM - 02:15 PM' batch_id='p1' total_pax=7

[CAPACITY CHECK] pickup_window='02:00 PM - 02:30 PM' coach_pax=108 vs total_coach_capacity=195

[BATCH p1 | 02:00 PM - 02:30 PM] Start cluster seed='little india' seed_pax=23
  Consider add 'ibis budget selagie' pax=2 closest_to='little india' dist=0.400 km | current_pax=23 -> would_be=25 | feasible_with_remaining_pool=True
  Consider add 'holiday inn little india' pax=4 closest_to='little india' dist=0.70

C:\Users\Sakthi\AppData\Local\Temp\ipykernel_13876\562398868.py:1055: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_one_pickup_window)
C:\Users\Sakthi\AppData\Local\Temp\ipykernel_13876\562398868.py:1114: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_one_return_window)
C:\Users\Sakthi\AppData\Local\Temp\ipykernel_13876\562398868.py:2579: FutureWarning: DataFrameGroupBy.apply operated on 


🎯 FINAL FILE GENERATED → 89_with_vehicle_reports.xlsx
✔️ Sheets: Final_Output, Batch_Summary, Vehicle_Tasks, Vehicle_Timeline
🎯 FINAL FILE GENERATED → 89.xlsx
